In [ ]:
import os
import sys

# --- FIX THE IMPORT ERROR ---
notebook_dir = os.getcwd() 
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root to sys.path: {project_root}")

# --- Imports from your project and external libraries ---
try:
    from src.dinov2_backbone import DinoV2Backbone
    from src.modules import Finalizer, build_fixation_selection_network, build_scanpath_network
    from src.layers import Bias
    from torch_scatter import scatter_mean
    print("Successfully imported project modules.")
except ImportError as e:
    print(f"FATAL: Still could not import modules. Error: {e}")
    print("Please ensure torch_scatter is installed.")
    
from src.data import convert_stimuli as convert_stimuli_mit

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import yaml
from types import SimpleNamespace
from pathlib import Path
import random
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import logging
import pysaliency
import pysaliency.external_datasets.mit
from pysaliency.dataset_config import validation_split

# --- Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
_logger = logging.getLogger("visualize_spade_notebook")

# --- Model Class Definitions (Copied from your script) ---
# We need to define the model classes here so we can instantiate them.
class SPADELayerNormDynamic(nn.Module):
    def __init__(self, norm_features, semantic_feature_channels, hidden_mlp_channels=128, eps=1e-12, kernel_size=3):
        super().__init__()
        self.norm_features = norm_features
        self.eps = eps
        padding = kernel_size // 2
        self.mlp_shared = nn.Sequential(
            nn.Conv2d(semantic_feature_channels, hidden_mlp_channels, kernel_size=kernel_size, padding=padding),
            nn.ReLU(inplace=True)
        )
        self.mlp_gamma = nn.Conv2d(hidden_mlp_channels, norm_features, kernel_size=kernel_size, padding=padding)
        self.mlp_beta = nn.Conv2d(hidden_mlp_channels, norm_features, kernel_size=kernel_size, padding=padding)

    def forward(self, x, painted_semantic_map):
        normalized_x = F.layer_norm(x, (self.norm_features, x.size(2), x.size(3)), eps=self.eps)
        semantic_map_resized = F.interpolate(painted_semantic_map, size=x.size()[2:], mode='bilinear', align_corners=False)
        shared_features = self.mlp_shared(semantic_map_resized)
        gamma_map = self.mlp_gamma(shared_features)
        beta_map = self.mlp_beta(shared_features)
        return normalized_x * (1 + gamma_map) + beta_map

class SaliencyNetworkSPADEDynamic(nn.Module):
    def __init__(self, input_channels_main_path, semantic_feature_channels_for_spade):
        super().__init__()
        self.spade_ln0 = SPADELayerNormDynamic(input_channels_main_path, semantic_feature_channels_for_spade)
        self.conv0 = nn.Conv2d(input_channels_main_path, 8, (1, 1), bias=False)
        self.bias0 = Bias(8); self.softplus0 = nn.Softplus()
        self.spade_ln1 = SPADELayerNormDynamic(8, semantic_feature_channels_for_spade)
        self.conv1 = nn.Conv2d(8, 16, (1, 1), bias=False)
        self.bias1 = Bias(16); self.softplus1 = nn.Softplus()
        self.spade_ln2 = SPADELayerNormDynamic(16, semantic_feature_channels_for_spade)
        self.conv2 = nn.Conv2d(16, 1, (1, 1), bias=False)
        self.bias2 = Bias(1); self.softplus2 = nn.Softplus()

    def forward(self, x_main_path, painted_semantic_map):
        h = self.spade_ln0(x_main_path, painted_semantic_map)
        h = self.softplus0(self.bias0(self.conv0(h)))
        h = self.spade_ln1(h, painted_semantic_map)
        h = self.softplus1(self.bias1(self.conv1(h)))
        h = self.spade_ln2(h, painted_semantic_map)
        h = self.softplus2(self.bias2(self.conv2(h)))
        return h

class DinoGazeSpade(nn.Module):
    def __init__(self, features_module, saliency_network, fixation_selection_network, **kwargs):
        super().__init__()
        self.features = features_module
        self.saliency_network = saliency_network
        self.fixation_selection_network = fixation_selection_network
        self.scanpath_network = kwargs.get('scanpath_network')
        self.semantic_feature_layer_idx = kwargs.get('semantic_feature_layer_idx', -1)
        self.num_total_segments = kwargs.get('num_total_sam_segments', 64)
        self.readout_factor = kwargs.get('readout_factor', 14)
        self.finalizer = Finalizer(
            sigma=kwargs.get('initial_sigma', 8.0),
            saliency_map_factor=kwargs.get('saliency_map_factor_finalizer', 4)
        )

    def _create_painted_semantic_map_vectorized(self, F_semantic_patches, raw_sam_pixel_segmap):
        B, C_dino, H_p, W_p = F_semantic_patches.shape
        _, H_img, W_img = raw_sam_pixel_segmap.shape
        device = F_semantic_patches.device
        segmap_at_feat_res = F.interpolate(raw_sam_pixel_segmap.unsqueeze(1).float(), size=(H_p, W_p), mode='nearest').long()
        flat_features = F_semantic_patches.permute(0, 2, 3, 1).reshape(-1, C_dino)
        flat_segmap_at_feat_res = segmap_at_feat_res.view(-1)
        batch_idx_tensor = torch.arange(B, device=device).view(B, 1).expand(-1, H_p * W_p).reshape(-1)
        global_segment_ids = batch_idx_tensor * self.num_total_segments + torch.clamp(flat_segmap_at_feat_res, 0, self.num_total_segments - 1)
        segment_avg_features = scatter_mean(src=flat_features, index=global_segment_ids, dim=0, dim_size=B * self.num_total_segments)
        segment_avg_features = torch.nan_to_num(segment_avg_features, nan=0.0)
        flat_pixel_segmap = raw_sam_pixel_segmap.view(B, -1)
        batch_idx_pixel_tensor = torch.arange(B, device=device).view(B, 1).expand(-1, H_img * W_img)
        global_pixel_ids = batch_idx_pixel_tensor.reshape(-1) * self.num_total_segments + torch.clamp(flat_pixel_segmap.view(-1), 0, self.num_total_segments - 1)
        painted_flat = segment_avg_features[global_pixel_ids]
        return painted_flat.view(B, H_img, W_img, C_dino).permute(0, 3, 1, 2)

    def forward(self, image, centerbias, segmentation_mask, **kwargs):
        with torch.no_grad():
            extracted_feature_maps = self.features(image)
        readout_h = math.ceil(image.shape[2] / self.readout_factor)
        readout_w = math.ceil(image.shape[3] / self.readout_factor)
        processed_features_list = [F.interpolate(f, size=(readout_h, readout_w), mode='bilinear') for f in extracted_feature_maps]
        concatenated_backbone_features = torch.cat(processed_features_list, dim=1)
        F_semantic_patches_from_dino = extracted_feature_maps[self.semantic_feature_layer_idx]
        S_painted_map_full_res = self._create_painted_semantic_map_vectorized(F_semantic_patches_from_dino, segmentation_mask)
        saliency_path_output = self.saliency_network(concatenated_backbone_features, S_painted_map_full_res)
        
        # For spatial-only visualization, scanpath output is None
        scanpath_path_output = None
        if self.scanpath_network is not None:
             B, _, H, W = saliency_path_output.shape
             scanpath_output_channels = 16
             scanpath_path_output = torch.zeros(B, scanpath_output_channels, H, W, device=image.device)
            
        combined_input_for_fixsel = (saliency_path_output, scanpath_path_output)
        final_readout_before_finalizer = self.fixation_selection_network(combined_input_for_fixsel)
        saliency_log_density = self.finalizer(final_readout_before_finalizer, centerbias)
        return saliency_log_density

In [ ]:
# ===================================================================
# --- MAIN EXECUTION CELL for SPADE MODEL (FINAL v12) ---
# This version correctly bypasses the fixation network AND inverts the
# resulting "energy map" for a correct and intuitive visualization.
# ===================================================================

# --- 1. USER CONFIGURATION ---
config_file_path = os.path.join(project_root, 'configs/mit_dinogaze_spade_dynamic_embedding_sam_64.yaml')
checkpoint_path = os.path.join(project_root, 'experiments/dinogaze_spade_dynamic_sam64_vitl14_v1/mit_scanpath_frozen_fold0_dinov2_vitl14_k64_lr0.0005/step-0056.pth')
output_path = os.path.join(project_root, 'output/spade_saliency_and_scanpath_visualization.png')
random_seed = 4
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD CONFIG and DATA ---
_logger.info("--- Loading config and dataset ---")
try:
    with open(config_file_path, 'r') as f:
        config = yaml.safe_load(f)
    dataset_root_dir = os.path.normpath(os.path.join(project_root, config.get('dataset_dir')))
    mask_root_dir = config.get('mit_all_mask_dir')
    fold = config.get('fold', 0)
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image: {os.path.basename(image_path)}")
    image_stem = Path(image_path).stem
    mask_path = os.path.join(project_root, mask_root_dir, f"{image_stem}.png")
    if not os.path.exists(mask_path): raise FileNotFoundError(f"Could not find mask at: {mask_path}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building DinoGazeSpade model...")
dino_backbone = DinoV2Backbone(layers=config['dino_layers_for_main_path'], model_name=config['dino_model_name'], patch_size=config['dino_patch_size'], freeze=True)
main_path_channels = len(config['dino_layers_for_main_path']) * dino_backbone.num_channels
semantic_path_channels = dino_backbone.num_channels
saliency_net = SaliencyNetworkSPADEDynamic(main_path_channels, semantic_path_channels)
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)
model = DinoGazeSpade(features_module=dino_backbone, saliency_network=saliency_net, fixation_selection_network=fixsel_net, scanpath_network=scanpath_net, semantic_feature_layer_idx=config['dino_semantic_feature_layer_idx'], num_total_segments=config['num_total_sam_segments'], initial_sigma=config['finalizer_initial_sigma'], readout_factor=config['dino_patch_size']).to(device)
_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE and MASK ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
mask_image = Image.open(mask_path).convert('L')
mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)

# --- 5. GENERATE SALIENCY MAP
_logger.info("Generating saliency map by tapping the pre-finalizer output...")
with torch.no_grad():
    extracted_feature_maps = model.features(image_tensor)
    readout_h = math.ceil(image_tensor.shape[2] / model.readout_factor)
    readout_w = math.ceil(image_tensor.shape[3] / model.readout_factor)
    processed_features_list = [F.interpolate(f, size=(readout_h, readout_w), mode='bilinear') for f in extracted_feature_maps]
    concatenated_backbone_features = torch.cat(processed_features_list, dim=1)
    F_semantic_patches_from_dino = extracted_feature_maps[model.semantic_feature_layer_idx]
    S_painted_map_full_res = model._create_painted_semantic_map_vectorized(F_semantic_patches_from_dino, mask_tensor)
    
    # This is the raw "energy map" from the saliency head.
    energy_grid = model.saliency_network(concatenated_backbone_features, S_painted_map_full_res)
    
    # Convert the "energy map" to a "saliency map" by negating it.
    # Now, high values will mean high saliency.
    saliency_grid = -energy_grid
    
    saliency_map_tensor = F.interpolate(saliency_grid, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)

saliency_map_np = saliency_map_tensor.squeeze().cpu().numpy()
_logger.info(f"Saliency map generated. Raw Range: [{saliency_map_np.min():.6f}, {saliency_map_np.max():.6f}]")

# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Saving visualization to: {output_path}")
# Perform min-max normalization for perfect visualization
saliency_min, saliency_max = saliency_map_np.min(), saliency_map_np.max()
heatmap_for_viz = (saliency_map_np - saliency_min) / (saliency_max - saliency_min) if saliency_max > saliency_min else saliency_map_np

fig, axes = plt.subplots(1, 5, figsize=(30, 6), dpi=120)
fig.suptitle('DinoGaze-SPADE Saliency Visualization (Corrected)', fontsize=16)

axes[0].imshow(original_image); axes[0].set_title('Original Image'); axes[0].axis('off')
axes[1].imshow(mask_np, cmap='nipy_spectral'); axes[1].set_title('Segmentation Mask'); axes[1].axis('off')
im = axes[2].imshow(heatmap_for_viz, cmap='hot'); axes[2].set_title('Saliency Heatmap'); axes[2].axis('off')
axes[3].imshow(original_image); axes[3].imshow(heatmap_for_viz, cmap='hot', alpha=0.5); axes[3].set_title('Overlay'); axes[3].axis('off')
axes[4].imshow(original_image); axes[4].set_title('Ground-Truth Scanpaths'); axes[4].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[4].plot(x_path, y_path, marker='o', linestyle='-', linewidth=1.5, markersize=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0:
    axes[4].legend()

fig.colorbar(im, ax=axes[2:4], orientation='horizontal', fraction=0.05, pad=0.08)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info("Visualization complete.")

In [ ]:
# ===================================================================
# --- COMBINED VISUALIZATION CELL (Sharp, Blurry, and SPADE Maps) ---
# This version visualizes everything: the two types of saliency maps
# and the internal gamma/beta modulation maps from the SPADE layers.
# ===================================================================

from src.dinov2_backbone import DinoV2Backbone

# --- 1. USER CONFIGURATION ---
config_file_path = os.path.join(project_root, 'configs/mit_dinogaze_spade_dynamic_embedding_sam_64.yaml')
checkpoint_path = os.path.join(project_root, 'experiments/dinogaze_spade_dynamic_sam64_vitl14_v1/mit_scanpath_frozen_fold0_dinov2_vitl14_k64_lr0.0005/step-0056.pth')
output_path = os.path.join(project_root, 'output/full_saliency_and_spade_visualization.png')
random_seed = 44 # Change this seed to get a different random image
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD CONFIG and DATA ---
_logger.info("--- Loading config and dataset ---")
try:
    with open(config_file_path, 'r') as f:
        config = yaml.safe_load(f)
    dataset_root_dir = os.path.normpath(os.path.join(project_root, config.get('dataset_dir')))
    mask_root_dir = config.get('mit_all_mask_dir')
    fold = config.get('fold', 0)
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image: {os.path.basename(image_path)}")
    image_stem = Path(image_path).stem
    mask_path = os.path.join(project_root, mask_root_dir, f"{image_stem}.png")
    if not os.path.exists(mask_path): raise FileNotFoundError(f"Could not find mask at: {mask_path}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building DinoGazeSpade model...")
dino_backbone = DinoV2Backbone(layers=config['dino_layers_for_main_path'], model_name=config['dino_model_name'], patch_size=config['dino_patch_size'], freeze=True)
main_path_channels = len(config['dino_layers_for_main_path']) * dino_backbone.num_channels
semantic_path_channels = dino_backbone.num_channels
saliency_net = SaliencyNetworkSPADEDynamic(main_path_channels, semantic_path_channels)
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)
model = DinoGazeSpade(features_module=dino_backbone, saliency_network=saliency_net, fixation_selection_network=fixsel_net, scanpath_network=scanpath_net, semantic_feature_layer_idx=config['dino_semantic_feature_layer_idx'], num_total_segments=config['num_total_sam_segments'], initial_sigma=config['finalizer_initial_sigma'], readout_factor=config['dino_patch_size']).to(device)
_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE and MASK ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
mask_image = Image.open(mask_path).convert('L')
mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)

# --- 5. SETUP HOOKS and GENERATE ALL MAPS ---
captured_maps = {}
def get_spade_hook(name):
    def hook(module, input, output):
        # We take the mean across the channel dimension for visualization
        processed_map = output.detach().squeeze(0).mean(dim=0).cpu().numpy()
        captured_maps[name] = processed_map
    return hook

hook_handles = []
spade_layers_to_hook = {
    'SPADE_0': model.saliency_network.spade_ln0,
    'SPADE_1': model.saliency_network.spade_ln1,
    'SPADE_2': model.saliency_network.spade_ln2,
}
for name, layer in spade_layers_to_hook.items():
    handle_g = layer.mlp_gamma.register_forward_hook(get_spade_hook(f'{name}_Gamma'))
    handle_b = layer.mlp_beta.register_forward_hook(get_spade_hook(f'{name}_Beta'))
    hook_handles.extend([handle_g, handle_b])
_logger.info(f"Attached {len(hook_handles)} hooks to SPADE layers.")

with torch.no_grad():
    # --- 5a. Generate "Sharp" Map (and trigger hooks) ---
    _logger.info("Generating 'Sharp' map and capturing SPADE maps...")
    extracted_feature_maps = model.features(image_tensor)
    readout_h = math.ceil(image_tensor.shape[2] / model.readout_factor)
    readout_w = math.ceil(image_tensor.shape[3] / model.readout_factor)
    processed_features_list = [F.interpolate(f, size=(readout_h, readout_w), mode='bilinear') for f in extracted_feature_maps]
    concatenated_backbone_features = torch.cat(processed_features_list, dim=1)
    F_semantic_patches_from_dino = extracted_feature_maps[model.semantic_feature_layer_idx]
    S_painted_map_full_res = model._create_painted_semantic_map_vectorized(F_semantic_patches_from_dino, mask_tensor)
    energy_grid = model.saliency_network(concatenated_backbone_features, S_painted_map_full_res)
    saliency_grid_sharp = -energy_grid
    saliency_map_tensor_sharp = F.interpolate(saliency_grid_sharp, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)
    saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()

    # --- 5b. Generate "Blurry" Map ---
    _logger.info("Generating 'Blurry' map (from full model output)...")
    class PassThrough(nn.Module):
        def forward(self, inputs, *_, **__): return inputs[0]
    model.fixation_selection_network = PassThrough()
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
    log_density_blurry = model(image_tensor, centerbias, segmentation_mask=mask_tensor)
    saliency_map_tensor_blurry = -log_density_blurry
    saliency_map_np_blurry = saliency_map_tensor_blurry.squeeze().cpu().numpy()

# --- 5c. Remove Hooks ---
for handle in hook_handles:
    handle.remove()
_logger.info("Removed all hooks.")

# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Creating combined visualization...")
# Min-max normalize all maps for visualization
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    if max_val > min_val:
        return (np_map - min_val) / (max_val - min_val)
    return np_map

heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)
heatmap_blurry = normalize_for_viz(saliency_map_np_blurry)
for key in captured_maps:
    captured_maps[key] = normalize_for_viz(captured_maps[key])

# Create a 4x4 grid for plotting
fig, axes = plt.subplots(4, 4, figsize=(24, 24), dpi=120)
fig.suptitle('DinoGaze-SPADE Full Analysis', fontsize=24)

# --- Row 1: General Info ---
axes[0, 0].imshow(original_image); axes[0, 0].set_title('Original Image', fontsize=16); axes[0, 0].axis('off')
axes[0, 1].imshow(mask_np, cmap='nipy_spectral'); axes[0, 1].set_title('Segmentation Mask', fontsize=16); axes[0, 1].axis('off')
axes[0, 2].imshow(original_image); axes[0, 2].set_title('Ground-Truth Scanpaths', fontsize=16); axes[0, 2].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 2].plot(x_path, y_path, marker='o', lw=1.5, ms=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0: axes[0, 2].legend()
axes[0, 3].axis('off') # Empty space

# --- Row 2: Saliency Map Visualizations ---
axes[1, 0].imshow(heatmap_sharp, cmap='hot'); axes[1, 0].set_title('Sharp Heatmap (Head Output)', fontsize=16); axes[1, 0].axis('off')
axes[1, 1].imshow(original_image); axes[1, 1].imshow(heatmap_sharp, cmap='hot', alpha=0.5); axes[1, 1].set_title('Sharp Overlay', fontsize=16); axes[1, 1].axis('off')
axes[1, 2].imshow(heatmap_blurry, cmap='hot'); axes[1, 2].set_title('Blurry Heatmap (Full Model)', fontsize=16); axes[1, 2].axis('off')
axes[1, 3].imshow(original_image); axes[1, 3].imshow(heatmap_blurry, cmap='hot', alpha=0.5); axes[1, 3].set_title('Blurry Overlay', fontsize=16); axes[1, 3].axis('off')

# --- Row 3: Gamma Modulation Maps (Feature Scaling) ---
# Use a diverging colormap to show positive/negative modulation
cmap_mod = 'viridis'
axes[2, 0].imshow(captured_maps['SPADE_0_Gamma'], cmap=cmap_mod); axes[2, 0].set_title('Gamma Map (SPADE 0)', fontsize=16); axes[2, 0].axis('off')
axes[2, 1].imshow(captured_maps['SPADE_1_Gamma'], cmap=cmap_mod); axes[2, 1].set_title('Gamma Map (SPADE 1)', fontsize=16); axes[2, 1].axis('off')
axes[2, 2].imshow(captured_maps['SPADE_2_Gamma'], cmap=cmap_mod); axes[2, 2].set_title('Gamma Map (SPADE 2)', fontsize=16); axes[2, 2].axis('off')
axes[2, 3].axis('off') # Empty space

# --- Row 4: Beta Modulation Maps (Feature Shifting) ---
axes[3, 0].imshow(-captured_maps['SPADE_0_Beta'], cmap=cmap_mod); axes[3, 0].set_title('Beta Map (SPADE 0)', fontsize=16); axes[3, 0].axis('off')
axes[3, 1].imshow(-captured_maps['SPADE_1_Beta'], cmap=cmap_mod); axes[3, 1].set_title('Beta Map (SPADE 1)', fontsize=16); axes[3, 1].axis('off')
axes[3, 2].imshow(-captured_maps['SPADE_2_Beta'], cmap=cmap_mod); axes[3, 2].set_title('Beta Map (SPADE 2)', fontsize=16); axes[3, 2].axis('off')
axes[3, 3].axis('off') # Empty space

plt.tight_layout(rect=[0, 0, 1, 0.97])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info(f"Full analysis visualization saved to: {output_path}")

In [ ]:
# ===================================================================
# --- COMBINED VISUALIZATION CELL for DinoGaze (non-SPADE) ---
# This single cell generates a full analysis in a 2x4 grid, comparing
# the raw saliency head output ('Sharp') vs. the full model's first
# fixation output ('Combined').
# ===================================================================

# --- Make sure all necessary imports are present ---
from src.dinov2_backbone import DinoV2Backbone
from src.dinogaze import build_saliency_network, build_scanpath_network, build_fixation_selection_network
from src.modules import DeepGazeIII
import torch.nn as nn
import torch.nn.functional as F
import math

# --- 1. USER CONFIGURATION ---
# Since there is no config file, we define key parameters here.
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'layers': [-3, -2, -1],
    'readout_factor': 7,
    'initial_sigma': 8.0
}
checkpoint_path = os.path.join(project_root, 'experiments/train_dinogaze_simple_test/mit_scanpath_frozen/crossval-10-0/final.pth')
dataset_dir_for_images = os.path.join(project_root, 'data/pysaliency_datasets') # <-- Path to where pysaliency stores datasets
output_path = os.path.join(project_root, 'output/dinogaze_non_spade_visualization.png')
random_seed = 44 # Change this seed to get a different random image
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD DATA ---
_logger.info("--- Loading dataset ---")
try:
    dataset_root_dir = os.path.normpath(dataset_dir_for_images)
    fold = 0
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image: {os.path.basename(image_path)}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building DeepGazeIII (non-SPADE) model...")
dino_backbone = DinoV2Backbone(layers=MODEL_PARAMS['layers'], model_name=MODEL_PARAMS['dino_model_name'], freeze=True)
C_in = len(MODEL_PARAMS['layers']) * dino_backbone.num_channels
saliency_net = build_saliency_network(C_in)
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)
model = DeepGazeIII(features=dino_backbone, saliency_network=saliency_net, scanpath_network=scanpath_net, fixation_selection_network=fixsel_net, readout_factor=MODEL_PARAMS['readout_factor'], initial_sigma=MODEL_PARAMS['initial_sigma']).to(device)
_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)

# --- 5. GENERATE BOTH SALIENCY MAPS (Manual Forward Pass) ---
with torch.no_grad():
    # --- 5a. Generate "Sharp" Map ---
    _logger.info("Generating 'Sharp' map (pre-Finalizer)...")
    features_list = model.features(image_tensor)
    concatenated_features = torch.cat(features_list, dim=1)
    energy_grid_sharp = model.saliency_network(concatenated_features)
    B, _, H, W = energy_grid_sharp.shape
    scanpath_shim = torch.zeros(B, 16, H, W, device=device)
    # This is the raw score grid before the Finalizer
    score_grid_before_finalizer = model.fixation_selection_network((energy_grid_sharp, scanpath_shim))
    # Upsample the score grid for visualization
    saliency_map_tensor_sharp = F.interpolate(score_grid_before_finalizer, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)
    saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()

    # --- 5b. Generate "Blurry" Map ---
    _logger.info("Generating 'Blurry' map (post-Finalizer)...")
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
    # Pass the pre-finalizer grid through the finalizer to get the final log-probability
    log_density_blurry = model.finalizer(score_grid_before_finalizer, centerbias)
    saliency_map_np_blurry = log_density_blurry.squeeze().cpu().numpy()

# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Creating combined 2x4 visualization...")
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map

heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)
heatmap_blurry = normalize_for_viz(saliency_map_np_blurry)

fig, axes = plt.subplots(2, 4, figsize=(26, 13), dpi=120)
fig.suptitle('DinoGaze (non-SPADE) Saliency Analysis: Head Output vs. Full Model', fontsize=20)

# --- Row 1: Inputs and Ground Truth ---
axes[0, 0].imshow(original_image); axes[0, 0].set_title('Original Image', fontsize=16); axes[0, 0].axis('off')
axes[0, 1].text(0.5, 0.5, 'No Segmentation Mask\nfor this Model', ha='center', va='center', fontsize=14, style='italic', color='gray'); axes[0, 1].set_title('Segmentation Mask', fontsize=16); axes[0, 1].axis('off')
axes[0, 2].imshow(original_image); axes[0, 2].set_title('Ground-Truth Scanpaths', fontsize=16); axes[0, 2].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 2].plot(x_path, y_path, marker='o', lw=1.5, ms=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0: axes[0, 2].legend()
axes[0, 3].axis('off')

# --- Row 2: Model Outputs ---
axes[1, 0].imshow(heatmap_sharp, cmap='hot'); axes[1, 0].set_title('Sharp Heatmap (Pre-Finalizer)', fontsize=16); axes[1, 0].axis('off')
axes[1, 1].imshow(original_image); axes[1, 1].imshow(heatmap_sharp, cmap='hot', alpha=0.5); axes[1, 1].set_title('Sharp Overlay', fontsize=16); axes[1, 1].axis('off')
axes[1, 2].imshow(heatmap_blurry, cmap='hot'); axes[1, 2].set_title('Blurry Heatmap (Post-Finalizer)', fontsize=16); axes[1, 2].axis('off')
axes[1, 3].imshow(original_image); axes[1, 3].imshow(heatmap_blurry, cmap='hot', alpha=0.5); axes[1, 3].set_title('Blurry Overlay', fontsize=16); axes[1, 3].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.97])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info(f"Combined visualization saved to: {output_path}")

In [ ]:
# ===================================================================
# --- COMBINED VISUALIZATION CELL (Sharp, Log-Density, SPADE Maps) ---
# This version visualizes the sharp saliency map, the final log-density
# map, and the internal gamma/beta modulation maps from SPADE layers.
# ===================================================================

from src.dinov2_backbone import DinoV2Backbone
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import random
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from PIL import Image
import os
import pysaliency

# --- 1. USER CONFIGURATION ---
config_file_path = os.path.join(project_root, 'configs/mit_dinogaze_spade_dynamic_embedding_sam_64.yaml')
checkpoint_path = os.path.join(project_root, 'experiments/dinogaze_spade_dynamic_sam64_vitl14_v1/mit_scanpath_frozen_fold0_dinov2_vitl14_k64_lr0.0005/step-0056.pth')
output_path = os.path.join(project_root, 'output/full_logdensity_and_spade_visualization.png')
random_seed = 44 # Change this seed to get a different random image
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD CONFIG and DATA ---
_logger.info("--- Loading config and dataset ---")
try:
    with open(config_file_path, 'r') as f:
        config = yaml.safe_load(f)
    dataset_root_dir = os.path.normpath(os.path.join(project_root, config.get('dataset_dir')))
    mask_root_dir = config.get('mit_all_mask_dir')
    fold = config.get('fold', 0)
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image: {os.path.basename(image_path)}")
    image_stem = Path(image_path).stem
    mask_path = os.path.join(project_root, mask_root_dir, f"{image_stem}.png")
    if not os.path.exists(mask_path): raise FileNotFoundError(f"Could not find mask at: {mask_path}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building DinoGazeSpade model...")
dino_backbone = DinoV2Backbone(layers=config['dino_layers_for_main_path'], model_name=config['dino_model_name'], patch_size=config['dino_patch_size'], freeze=True)
main_path_channels = len(config['dino_layers_for_main_path']) * dino_backbone.num_channels
semantic_path_channels = dino_backbone.num_channels
saliency_net = SaliencyNetworkSPADEDynamic(main_path_channels, semantic_path_channels)
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)
model = DinoGazeSpade(features_module=dino_backbone, saliency_network=saliency_net, fixation_selection_network=fixsel_net, scanpath_network=scanpath_net, semantic_feature_layer_idx=config['dino_semantic_feature_layer_idx'], num_total_segments=config['num_total_sam_segments'], initial_sigma=config['finalizer_initial_sigma'], readout_factor=config['dino_patch_size']).to(device)
_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE and MASK ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
mask_image = Image.open(mask_path).convert('L')
mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)

# --- 5. SETUP HOOKS and GENERATE ALL MAPS ---
captured_maps = {}
def get_spade_hook(name):
    def hook(module, input, output):
        processed_map = output.detach().squeeze(0).mean(dim=0).cpu().numpy()
        captured_maps[name] = processed_map
    return hook

hook_handles = []
spade_layers_to_hook = {
    'SPADE_0': model.saliency_network.spade_ln0,
    'SPADE_1': model.saliency_network.spade_ln1,
    'SPADE_2': model.saliency_network.spade_ln2,
}
for name, layer in spade_layers_to_hook.items():
    handle_g = layer.mlp_gamma.register_forward_hook(get_spade_hook(f'{name}_Gamma'))
    handle_b = layer.mlp_beta.register_forward_hook(get_spade_hook(f'{name}_Beta'))
    hook_handles.extend([handle_g, handle_b])
_logger.info(f"Attached {len(hook_handles)} hooks to SPADE layers.")

with torch.no_grad():
    # --- 5a. Generate "Sharp" Map (negated energy) ---
    _logger.info("Generating 'Sharp' map and capturing SPADE maps...")
    extracted_feature_maps = model.features(image_tensor)
    readout_h = math.ceil(image_tensor.shape[2] / model.readout_factor)
    readout_w = math.ceil(image_tensor.shape[3] / model.readout_factor)
    processed_features_list = [F.interpolate(f, size=(readout_h, readout_w), mode='bilinear') for f in extracted_feature_maps]
    concatenated_backbone_features = torch.cat(processed_features_list, dim=1)
    F_semantic_patches_from_dino = extracted_feature_maps[model.semantic_feature_layer_idx]
    S_painted_map_full_res = model._create_painted_semantic_map_vectorized(F_semantic_patches_from_dino, mask_tensor)
    energy_grid = model.saliency_network(concatenated_backbone_features, S_painted_map_full_res)
    # For sharp map, we still visualize as "saliency" (higher is better)
    saliency_grid_sharp = -energy_grid
    saliency_map_tensor_sharp = F.interpolate(saliency_grid_sharp, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)
    saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()

    # --- 5b. Generate Log-Density Map (Final Model Output) ---
    _logger.info("Generating final Log-Density map...")
    # Temporarily replace fixation selection network to get first-fixation map
    original_fixsel_net = model.fixation_selection_network
    class PassThrough(nn.Module):
        def forward(self, inputs, *_, **__): return inputs[0]
    model.fixation_selection_network = PassThrough()
    centerbias_shim = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
    
    # The direct output of the model IS the log-density
    log_density_map_tensor = model(image_tensor, centerbias_shim, segmentation_mask=mask_tensor)
    log_density_map_np = log_density_map_tensor.squeeze().cpu().numpy()
    
    # Restore original network
    model.fixation_selection_network = original_fixsel_net

# --- 5c. Remove Hooks ---
for handle in hook_handles:
    handle.remove()
_logger.info("Removed all hooks.")


# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Creating combined visualization...")

# Normalization function for visualization (scales 0 to 1)
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    if max_val > min_val:
        return (np_map - min_val) / (max_val - min_val)
    return np_map

# Normalize the sharp map for 'hot' colormap
heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)

# The log-density map does NOT need normalization before plotting with a colorbar.
# We will use its real values.
# The gamma/beta maps are normalized for visualization.
for key in captured_maps:
    captured_maps[key] = normalize_for_viz(captured_maps[key])


# --- Create Plot ---
fig, axes = plt.subplots(4, 4, figsize=(24, 24), dpi=120)
fig.suptitle('DinoGaze-SPADE Full Analysis (with Log-Density)', fontsize=24)

# --- Row 1: General Info ---
axes[0, 0].imshow(original_image); axes[0, 0].set_title('Original Image', fontsize=16); axes[0, 0].axis('off')
axes[0, 1].imshow(mask_np, cmap='nipy_spectral'); axes[0, 1].set_title('Segmentation Mask', fontsize=16); axes[0, 1].axis('off')
axes[0, 2].imshow(original_image); axes[0, 2].set_title('Ground-Truth Scanpaths', fontsize=16); axes[0, 2].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 2].plot(x_path, y_path, marker='o', lw=1.5, ms=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0: axes[0, 2].legend()
axes[0, 3].axis('off') 

# --- Row 2: Saliency & Log-Density Maps ---
# Sharp map is visualized as saliency (hot is high)
axes[1, 0].imshow(heatmap_sharp, cmap='hot'); axes[1, 0].set_title('Sharp Heatmap (-Energy)', fontsize=16); axes[1, 0].axis('off')
axes[1, 1].imshow(original_image); axes[1, 1].imshow(heatmap_sharp, cmap='hot', alpha=0.5); axes[1, 1].set_title('Sharp Overlay', fontsize=16); axes[1, 1].axis('off')

# Log-Density map is visualized with a sequential colormap where bright is high (less negative)
# We plot the raw values and add a colorbar to show the scale.
log_density_plot = axes[1, 2].imshow(log_density_map_np, cmap='viridis') 
axes[1, 2].set_title('Final Log-Density Map', fontsize=16); axes[1, 2].axis('off')
fig.colorbar(log_density_plot, ax=axes[1, 2], shrink=0.8) # Add colorbar for scale

axes[1, 3].imshow(original_image); axes[1, 3].imshow(log_density_map_np, cmap='viridis', alpha=0.5); axes[1, 3].set_title('Log-Density Overlay', fontsize=16); axes[1, 3].axis('off')

# --- Row 3: Gamma Modulation Maps (Feature Scaling) ---
# Use a diverging colormap to show positive/negative modulation
cmap_mod = 'coolwarm' 
axes[2, 0].imshow(captured_maps['SPADE_0_Gamma'], cmap=cmap_mod); axes[2, 0].set_title('Gamma Map (SPADE 0)', fontsize=16); axes[2, 0].axis('off')
axes[2, 1].imshow(captured_maps['SPADE_1_Gamma'], cmap=cmap_mod); axes[2, 1].set_title('Gamma Map (SPADE 1)', fontsize=16); axes[2, 1].axis('off')
axes[2, 2].imshow(captured_maps['SPADE_2_Gamma'], cmap=cmap_mod); axes[2, 2].set_title('Gamma Map (SPADE 2)', fontsize=16); axes[2, 2].axis('off')
axes[2, 3].axis('off')

# --- Row 4: Beta Modulation Maps (Feature Shifting) ---
axes[3, 0].imshow(captured_maps['SPADE_0_Beta'], cmap=cmap_mod); axes[3, 0].set_title('Beta Map (SPADE 0)', fontsize=16); axes[3, 0].axis('off')
axes[3, 1].imshow(captured_maps['SPADE_1_Beta'], cmap=cmap_mod); axes[3, 1].set_title('Beta Map (SPADE 1)', fontsize=16); axes[3, 1].axis('off')
axes[3, 2].imshow(captured_maps['SPADE_2_Beta'], cmap=cmap_mod); axes[3, 2].set_title('Beta Map (SPADE 2)', fontsize=16); axes[3, 2].axis('off')
axes[3, 3].axis('off')

# --- Finalize and Save ---
plt.tight_layout(rect=[0, 0, 1, 0.97])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info(f"Full analysis visualization saved to: {output_path}")

In [ ]:
# ===================================================================
# --- COMBINED VISUALIZATION CELL for DinoGaze (non-SPADE) ---
# This single cell generates a full analysis in a 2x4 grid, comparing
# the raw saliency head output ('Sharp') vs. the final log-density map.
# ===================================================================

# --- Make sure all necessary imports are present ---
from src.dinov2_backbone import DinoV2Backbone
from src.dinogaze import build_saliency_network, build_scanpath_network, build_fixation_selection_network
from src.modules import DeepGazeIII
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import numpy as np
import random
from PIL import Image
from pathlib import Path
import os
import pysaliency
import matplotlib.pyplot as plt

# --- 1. USER CONFIGURATION ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'layers': [-3, -2, -1],
    'readout_factor': 7,
    'initial_sigma': 8.0
}
# IMPORTANT: Update this to a valid checkpoint for the non-SPADE model
checkpoint_path = os.path.join(project_root, 'experiments/train_dinogaze_simple_test/mit_scanpath_frozen/crossval-10-0/final.pth') 
dataset_dir_for_images = os.path.join(project_root, 'data/pysaliency_datasets') 
output_path = os.path.join(project_root, 'output/dinogaze_non_spade_logdensity_visualization.png')
random_seed = 44 # Use the same seed as the SPADE script for a direct comparison
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD DATA ---
_logger.info("--- Loading dataset ---")
try:
    dataset_root_dir = os.path.normpath(dataset_dir_for_images)
    fold = 0
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image for non-SPADE: {os.path.basename(image_path)}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building DeepGazeIII (non-SPADE) model...")
dino_backbone = DinoV2Backbone(layers=MODEL_PARAMS['layers'], model_name=MODEL_PARAMS['dino_model_name'], freeze=True)
C_in = len(MODEL_PARAMS['layers']) * dino_backbone.num_channels
saliency_net = build_saliency_network(C_in)
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)
model = DeepGazeIII(features=dino_backbone, saliency_network=saliency_net, scanpath_network=scanpath_net, fixation_selection_network=fixsel_net, readout_factor=MODEL_PARAMS['readout_factor'], initial_sigma=MODEL_PARAMS['initial_sigma']).to(device)
_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)

# --- 5. GENERATE MAPS (Manual Forward Pass) ---
with torch.no_grad():
    # --- 5a. Generate "Sharp" Map (raw score from head) ---
    _logger.info("Generating 'Sharp' map (pre-Finalizer)...")
    features_list = model.features(image_tensor)
    concatenated_features = torch.cat(features_list, dim=1)
    saliency_path_output = model.saliency_network(concatenated_features)
    
    # Create a zero-tensor shim for the scanpath input (for first fixation)
    B, _, H, W = saliency_path_output.shape
    scanpath_shim = torch.zeros(B, 16, H, W, device=device)
    
    # This is the raw score grid before the Finalizer
    score_grid_before_finalizer = model.fixation_selection_network((saliency_path_output, scanpath_shim))
    
    # Upsample the score grid for visualization
    saliency_map_tensor_sharp = F.interpolate(score_grid_before_finalizer, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)
    saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()

    # --- 5b. Generate Log-Density Map (Final Model Output) ---
    _logger.info("Generating final Log-Density map (post-Finalizer)...")
    centerbias_shim = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
    
    # Pass the pre-finalizer grid through the finalizer to get the log-density
    log_density_map_tensor = model.finalizer(score_grid_before_finalizer, centerbias_shim)
    log_density_map_np = log_density_map_tensor.squeeze().cpu().numpy()

# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Creating combined 2x4 visualization...")
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map

# Normalize the sharp map for 'hot' colormap visualization
heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)

# Log-density map does not need normalization for plotting with a colorbar.

fig, axes = plt.subplots(2, 4, figsize=(26, 13), dpi=120)
fig.suptitle('DinoGaze (non-SPADE) Analysis (with Log-Density)', fontsize=20)

# --- Row 1: Inputs and Ground Truth ---
axes[0, 0].imshow(original_image); axes[0, 0].set_title('Original Image', fontsize=16); axes[0, 0].axis('off')
axes[0, 1].text(0.5, 0.5, 'No Segmentation Mask\nfor this Model', ha='center', va='center', fontsize=14, style='italic', color='gray'); axes[0, 1].set_title('Context Input', fontsize=16); axes[0, 1].axis('off')
axes[0, 2].imshow(original_image); axes[0, 2].set_title('Ground-Truth Scanpaths', fontsize=16); axes[0, 2].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 2].plot(x_path, y_path, marker='o', lw=1.5, ms=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0: axes[0, 2].legend()
axes[0, 3].axis('off')

# --- Row 2: Model Outputs ---
# Sharp map is visualized as saliency (hot is high)
axes[1, 0].imshow(heatmap_sharp, cmap='hot'); axes[1, 0].set_title('Sharp Heatmap (Head Score)', fontsize=16); axes[1, 0].axis('off')
axes[1, 1].imshow(original_image); axes[1, 1].imshow(heatmap_sharp, cmap='hot', alpha=0.5); axes[1, 1].set_title('Sharp Overlay', fontsize=16); axes[1, 1].axis('off')

# Log-Density map is visualized with a sequential colormap where bright is high (less negative)
log_density_plot = axes[1, 2].imshow(log_density_map_np, cmap='viridis')
axes[1, 2].set_title('Final Log-Density Map', fontsize=16); axes[1, 2].axis('off')
fig.colorbar(log_density_plot, ax=axes[1, 2], shrink=0.8) # Add colorbar for scale

axes[1, 3].imshow(original_image); axes[1, 3].imshow(log_density_map_np, cmap='viridis', alpha=0.5); axes[1, 3].set_title('Log-Density Overlay', fontsize=16); axes[1, 3].axis('off')


# --- Finalize and Save ---
plt.tight_layout(rect=[0, 0, 1, 0.97])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info(f"Combined visualization saved to: {output_path}")

In [ ]:
# ===================================================================
# --- COMBINED VISUALIZATION CELL for Original DeepGaze III (DenseNet) ---
# This single cell generates a full analysis in a 2x4 grid, comparing
# the raw pre-Finalizer output ('Sharp') vs. the full model's final
# output ('Blurry').
# ===================================================================

# --- Make sure all necessary imports are present ---
# We need to import the specific DenseNet backbone
from DeepGaze.deepgaze_pytorch.features.densenet import RGBDenseNet201
from src.modules import DeepGazeIII, FeatureExtractor
from src.layers import Bias, LayerNorm, LayerNormMultiInput, Conv2dMultiInput, FlexibleScanpathHistoryEncoding
from src.dinogaze import build_saliency_network, build_scanpath_network, build_fixation_selection_network
import torch.nn as nn
import torch.nn.functional as F
import math

# --- 1. USER CONFIGURATION ---
# Define key parameters for the original DeepGaze III model architecture
MODEL_PARAMS = {
    'densenet_feature_nodes': [
        '1.features.denseblock4.denselayer32.norm1',
        '1.features.denseblock4.denselayer32.conv1',
        '1.features.denseblock4.denselayer31.conv2',
    ],
    'saliency_input_channels': 2048,
    'readout_factor': 4,
    'initial_sigma': 8.0,
    'downsample': 1.0, 
}
checkpoint_path = os.path.join(project_root, 'experiments/train_deepgaze3/MIT1003_scanpath_partially_frozen_saliency_network/crossval-10-0/step-0014.pth') # <-- IMPORTANT: UPDATE THIS PATH
dataset_dir_for_images = os.path.join(project_root, 'data/pysaliency_datasets')
output_path = os.path.join(project_root, 'output/original_deepgaze_comparison.png')
random_seed = 44 # Change this seed to get a different random image
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
_logger.info(f"Using device: {device}")

# --- 2. LOAD DATA ---
_logger.info("--- Loading dataset ---")
try:
    dataset_root_dir = os.path.normpath(dataset_dir_for_images)
    fold = 0 # Assume fold 0
    mit_stimuli, mit_fixations = pysaliency.external_datasets.mit.get_mit1003_with_initial_fixation(location=dataset_root_dir, replace_initial_invalid_fixations=True)
    val_stim, _ = validation_split(mit_stimuli, mit_fixations, crossval_folds=10, fold_no=fold)
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    _logger.info(f"Randomly selected image: {os.path.basename(image_path)}")
    stimulus_index = mit_stimuli.filenames.index(image_path)
    image_scanpaths = mit_fixations.scanpaths[mit_fixations.scanpaths.n == stimulus_index]
    list_of_x_paths, list_of_y_paths = image_scanpaths.xs, image_scanpaths.ys
    _logger.info(f"Found {len(list_of_x_paths)} scanpaths for this image.")
except Exception as e:
    _logger.error(f"Failed during data pipeline setup: {e}")
    raise

# --- 3. BUILD and LOAD MODEL ---
_logger.info("Building Original DeepGazeIII (DenseNet) model...")
# Build the model exactly as specified in the training script
densenet_base = RGBDenseNet201()
features_module = FeatureExtractor(densenet_base, MODEL_PARAMS['densenet_feature_nodes'])

saliency_net = build_saliency_network(MODEL_PARAMS['saliency_input_channels'])
scanpath_net = build_scanpath_network()
fixsel_net = build_fixation_selection_network(scanpath_features=16)

model = DeepGazeIII(
    features=features_module,
    saliency_network=saliency_net,
    scanpath_network=scanpath_net,
    fixation_selection_network=fixsel_net,
    downsample=MODEL_PARAMS['downsample'],
    readout_factor=MODEL_PARAMS['readout_factor'],
    initial_sigma=MODEL_PARAMS['initial_sigma']
).to(device)

_logger.info(f"Loading checkpoint: {checkpoint_path}")
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_state_dict = checkpoint_data.get('model', checkpoint_data)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=True)
model.eval()
_logger.info("Model weights loaded successfully.")

# --- 4. PREPROCESS IMAGE ---
original_image = Image.open(image_path).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(device)

# --- 5. GENERATE MAPS ---
with torch.no_grad():
    _logger.info("Generating 'Sharp' and 'Blurry' maps...")
    orig_shape = image_tensor.shape
    x = F.interpolate(image_tensor, scale_factor=1 / model.downsample)
    features_list = model.features(x)
    readout_shape = [math.ceil(orig_shape[2] / model.downsample / model.readout_factor), math.ceil(orig_shape[3] / model.downsample / model.readout_factor)]
    x_resized = [F.interpolate(item, readout_shape) for item in features_list]
    concatenated_features = torch.cat(x_resized, dim=1)
    saliency_grid = model.saliency_network(concatenated_features)
    B, _, H, W = saliency_grid.shape
    scanpath_shim = torch.zeros(B, 16, H, W, device=device)
    score_grid_before_finalizer = model.fixation_selection_network((saliency_grid, scanpath_shim))
    
    _logger.info(f"--> TRUE Readout Grid Size (H x W): {score_grid_before_finalizer.shape[2]} x {score_grid_before_finalizer.shape[3]}")
    
    # --- 5a. Generate "Blocky" Sharp Map with 'nearest' interpolation ---
    saliency_map_tensor_sharp_blocky = F.interpolate(score_grid_before_finalizer, size=image_tensor.shape[2:], mode='bilinear')
    saliency_map_np_sharp_blocky = saliency_map_tensor_sharp_blocky.squeeze().cpu().numpy()

    # --- 5b. Generate "Blurry" Map from the Finalizer ---
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
    log_density_blurry = model.finalizer(score_grid_before_finalizer, centerbias)
    saliency_map_np_blurry = log_density_blurry.squeeze().cpu().numpy()

# --- 6. VISUALIZE and SAVE ---
_logger.info(f"Creating visualization...")
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map

heatmap_sharp_blocky = normalize_for_viz(saliency_map_np_sharp_blocky)
heatmap_blurry = normalize_for_viz(saliency_map_np_blurry)

fig, axes = plt.subplots(2, 3, figsize=(21, 14), dpi=120)
fig.suptitle('DeepGaze III Analysis: The Readout Grid Revealed', fontsize=20)

# --- Row 1: Inputs and Ground Truth ---
axes[0, 0].imshow(original_image); axes[0, 0].set_title('Original Image'); axes[0, 0].axis('off')
axes[0, 1].imshow(original_image); axes[0, 1].set_title('Ground-Truth Scanpaths'); axes[0, 1].axis('off')
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 1].plot(x_path, y_path, marker='o', lw=1.5, ms=5, alpha=0.7, label=f'Subj {i+1}' if i < 3 else None)
if len(list_of_x_paths) > 0: axes[0, 1].legend()
axes[0, 2].axis('off') # Empty panel

# --- Row 2: Model Outputs ---
axes[1, 0].imshow(heatmap_sharp_blocky, cmap='hot'); axes[1, 0].set_title('"Sharp" Map (Nearest Upsample)'); axes[1, 0].axis('off')
axes[1, 1].imshow(heatmap_blurry, cmap='hot'); axes[1, 1].set_title('"Blurry" Map (Finalizer Output)'); axes[1, 1].axis('off')
axes[1, 2].imshow(original_image); axes[1, 2].imshow(heatmap_blurry, cmap='hot', alpha=0.5); axes[1, 2].set_title('Blurry Overlay'); axes[1, 2].axis('off')


plt.tight_layout(rect=[0, 0, 1, 0.96])
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close()
_logger.info(f"Visualization saved to: {output_path}")

In [ ]:
# = ==================================================================
# --- INTERACTIVE EMBEDDING EXPLORATION CELL (DEFINITIVE) ---
# This version robustly finds the correct data cache by using the
# checkpoint path, and uses '%matplotlib widget' for interactivity.
# ===================================================================

# --- Make sure all necessary imports are present ---
import numpy as np
import torch
import random
from tqdm.notebook import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, clear_output
import umap
import cloudpickle as cpickle
import torch.nn.functional as F



# This magic command is essential for interactive plots in VS Code / JupyterLab.
# It should be at the top of the cell.
%matplotlib ipympl 

# --- 1. CONFIGURATION ---
NUM_IMAGES_TO_PROCESS = 100 
N_NEIGHBORS_TO_SHOW = 9

checkpoint_path = os.path.join(project_root, 'experiments/mit_spatial_finetune/dinogaze_spade_v1_fold0/final_best_val.pth') # <-- IMPORTANT: UPDATE THIS PATH

# --- 2. HELPER FUNCTION: EMBEDDING EXTRACTOR (Unchanged) ---
def extract_segment_embeddings(model, image_tensor, mask_tensor):
    with torch.no_grad():
        extracted_feature_maps = model.features(image_tensor)
        F_semantic_patches = extracted_feature_maps[model.semantic_feature_layer_idx]
        B, C_dino, H_p, W_p = F_semantic_patches.shape; device = F_semantic_patches.device
        segmap_at_feat_res = F.interpolate(mask_tensor.unsqueeze(1).float(), size=(H_p, W_p), mode='nearest').long()
        flat_features = F_semantic_patches.permute(0, 2, 3, 1).reshape(-1, C_dino)
        flat_segmap_at_feat_res = segmap_at_feat_res.view(-1)
        batch_idx_tensor = torch.arange(B, device=device).view(B, 1).expand(-1, H_p * W_p).reshape(-1)
        global_segment_ids = batch_idx_tensor * model.num_total_segments + torch.clamp(flat_segmap_at_feat_res, 0, model.num_total_segments - 1)
        segment_avg_features = scatter_mean(src=flat_features, index=global_segment_ids, dim=0, dim_size=B * model.num_total_segments)
        segment_avg_features = torch.nan_to_num(segment_avg_features, nan=0.0)
        return segment_avg_features.view(B, model.num_total_segments, C_dino)

# --- 3. DATA COLLECTION LOOP (with ROBUST cache path finding) ---
_logger.info(f"Starting embedding extraction from {NUM_IMAGES_TO_PROCESS} random images...")
all_embeddings, metadata = [], []

checkpoint_path_obj = Path(checkpoint_path)
correct_train_dir_base = checkpoint_path_obj.parent.parent
_logger.info(f"Inferred base experiment directory: {correct_train_dir_base}")
mit_converted_data_path = correct_train_dir_base / f"MIT1003_converted_dinogaze_{config.get('dino_model_name')}"
stimuli_cache_path = mit_converted_data_path / "stimuli.pkl"
if not stimuli_cache_path.exists():
    raise FileNotFoundError(f"Could not find processed stimuli cache at: {stimuli_cache_path}.")
with open(stimuli_cache_path, "rb") as f:
    stimuli_resized_object = cpickle.load(f)

stimuli_dir = mit_converted_data_path / "stimuli"
reconstructed_filenames = [str(stimuli_dir / Path(f).name) for f in stimuli_resized_object.filenames]
stimuli_resized = pysaliency.FileStimuli(filenames=reconstructed_filenames)
_logger.info(f"Reconstructed {len(reconstructed_filenames)} valid image paths.")

val_stim_resized, _ = validation_split(stimuli_resized, mit_fixations, crossval_folds=10, fold_no=config.get('fold',0))
image_files = val_stim_resized.filenames
random.shuffle(image_files)

for image_path in tqdm(image_files[:NUM_IMAGES_TO_PROCESS]):
    try:
        img_pil = Image.open(image_path).convert("RGB")
        img_np = np.array(img_pil)
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
        mask_stem = Path(image_path).stem
        mask_path = os.path.join(project_root, config.get('mit_all_mask_dir'), f"{mask_stem}.png")
        mask_pil = Image.open(mask_path).convert("L")
        if mask_pil.size != img_pil.size:
            mask_pil = mask_pil.resize(img_pil.size, Image.Resampling.NEAREST)
        mask_np = np.array(mask_pil)
        mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)
        embeddings_for_image = extract_segment_embeddings(model, img_tensor, mask_tensor).squeeze(0)
        present_segments = torch.unique(mask_tensor).cpu().numpy()
        for seg_id in present_segments:
            if seg_id >= model.num_total_segments: continue
            embedding = embeddings_for_image[seg_id].cpu().numpy()
            if np.all(embedding == 0): continue
            all_embeddings.append(embedding)
            segment_pixels = np.where(mask_np == seg_id)
            if segment_pixels[0].size > 0:
                ymin, ymax, xmin, xmax = segment_pixels[0].min(), segment_pixels[0].max(), segment_pixels[1].min(), segment_pixels[1].max()
                segment_img_rgba = Image.fromarray(img_np).convert("RGBA")
                mask_for_alpha = np.zeros_like(mask_np, dtype=np.uint8)
                mask_for_alpha[segment_pixels] = 255
                segment_img_rgba.putalpha(Image.fromarray(mask_for_alpha))
                cropped_segment = segment_img_rgba.crop((xmin, ymin, xmax, ymax))
                metadata.append({'image_path': image_path, 'segment_id': seg_id, 'segment_image': cropped_segment})
    except Exception as e:
        _logger.warning(f"Skipping {os.path.basename(image_path)} due to error: {e}")

# --- 4. ROBUST DIMENSIONALITY REDUCTION ---
MIN_SAMPLES_FOR_UMAP = 20
if len(all_embeddings) < MIN_SAMPLES_FOR_UMAP:
    _logger.error(f"Not enough valid segment embeddings extracted ({len(all_embeddings)} found). Need at least {MIN_SAMPLES_FOR_UMAP} for UMAP.")
else:
    all_embeddings = np.array(all_embeddings)
    _logger.info(f"Extracted {len(all_embeddings)} total segment embeddings.")
    _logger.info("Performing dimensionality reduction with UMAP...")
    n_neighbors_for_umap = min(15, len(all_embeddings) - 1)
    reducer = umap.UMAP(n_neighbors=n_neighbors_for_umap, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(all_embeddings)
    _logger.info("UMAP fitting complete.")

    # --- 5. INTERACTIVE VISUALIZATION ---
    _logger.info("Creating interactive plot. Click on a point to explore neighbors!")
    nn_model = NearestNeighbors(n_neighbors=min(N_NEIGHBORS_TO_SHOW, len(embeddings_2d)), metric='euclidean')
    nn_model.fit(embeddings_2d)

    fig_main, ax_main = plt.subplots(figsize=(10, 8))
    colors = embeddings_2d[:, 1]
    scatter = ax_main.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=5, alpha=0.6, c=colors, cmap='viridis')
    ax_main.set_title("UMAP Projection of Segment Embeddings\n(Click to Explore)")
    
    # The output widget will hold the neighbor plots
    out = widgets.Output()

    def on_click(event):
        if event.inaxes != ax_main: return
        click_coords = np.array([[event.xdata, event.ydata]])
        distances, indices = nn_model.kneighbors(click_coords)
        with out:
            clear_output(wait=True)
            grid_cols, grid_rows = 3, int(np.ceil(N_NEIGHBORS_TO_SHOW / 3))
            fig_neighbors, axes_neighbors = plt.subplots(grid_rows, grid_cols, figsize=(12, 4 * grid_rows))
            # Handle case where there's only one row
            if grid_rows == 1:
                axes_neighbors = np.array([axes_neighbors])
            axes_neighbors = axes_neighbors.flatten()
            fig_neighbors.suptitle(f"Nearest Neighbors in Embedding Space", fontsize=16)
            for i, idx in enumerate(indices[0]):
                meta = metadata[idx]
                ax = axes_neighbors[i]
                ax.imshow(meta['segment_image'])
                ax.set_title(f"{os.path.basename(meta['image_path'])}\nSegment ID: {meta['segment_id']}", fontsize=8)
                ax.axis('off')
            for i in range(len(indices[0]), len(axes_neighbors)):
                axes_neighbors[i].axis('off')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()

    fig_main.canvas.mpl_connect('button_press_event', on_click)
    
    # --- THIS IS THE FIX ---
    # Display the output widget directly. The main figure is already
    # handled by the %matplotlib widget backend and will appear above it.
    display(out)
    # --- END OF FIX ---

In [ ]:
# ===================================================================
# --- INTERACTIVE EMBEDDING EXPLORATION (with Style Normalization) ---
# This definitive version normalizes out the "image style" from embeddings
# to reveal purer semantic clusters.
# ===================================================================

# --- Make sure all necessary imports are present ---
import numpy as np
import torch
import random
from tqdm.notebook import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, clear_output
import umap
import cloudpickle as cpickle
import torch.nn.functional as F

%matplotlib widget

# --- 1. CONFIGURATION ---
NUM_IMAGES_TO_PROCESS = 1000 # Using 1000 to get a rich embedding space
N_NEIGHBORS_TO_SHOW = 9    

# --- 2. HELPER FUNCTION: EMBEDDING EXTRACTOR (Unchanged) ---
def extract_segment_embeddings(model, image_tensor, mask_tensor):
    with torch.no_grad():
        extracted_feature_maps = model.features(image_tensor)
        F_semantic_patches = extracted_feature_maps[model.semantic_feature_layer_idx]
        B, C_dino, H_p, W_p = F_semantic_patches.shape; device = F_semantic_patches.device
        segmap_at_feat_res = F.interpolate(mask_tensor.unsqueeze(1).float(), size=(H_p, W_p), mode='nearest').long()
        flat_features = F_semantic_patches.permute(0, 2, 3, 1).reshape(-1, C_dino)
        flat_segmap_at_feat_res = segmap_at_feat_res.view(-1)
        batch_idx_tensor = torch.arange(B, device=device).view(B, 1).expand(-1, H_p * W_p).reshape(-1)
        global_segment_ids = batch_idx_tensor * model.num_total_segments + torch.clamp(flat_segmap_at_feat_res, 0, model.num_total_segments - 1)
        segment_avg_features = scatter_mean(src=flat_features, index=global_segment_ids, dim=0, dim_size=B * model.num_total_segments)
        segment_avg_features = torch.nan_to_num(segment_avg_features, nan=0.0)
        return segment_avg_features.view(B, model.num_total_segments, C_dino)

# --- 3. DATA COLLECTION LOOP (with Style Normalization) ---
_logger.info(f"Starting embedding extraction from {NUM_IMAGES_TO_PROCESS} random images...")
all_embeddings, metadata = [], []

checkpoint_path_obj = Path(checkpoint_path)
correct_train_dir_base = checkpoint_path_obj.parent.parent
mit_converted_data_path = correct_train_dir_base / f"MIT1003_converted_dinogaze_{config.get('dino_model_name')}"
stimuli_cache_path = mit_converted_data_path / "stimuli.pkl"
if not stimuli_cache_path.exists():
    raise FileNotFoundError(f"Could not find processed stimuli cache at: {stimuli_cache_path}.")
with open(stimuli_cache_path, "rb") as f:
    stimuli_resized_object = cpickle.load(f)
stimuli_dir = mit_converted_data_path / "stimuli"
reconstructed_filenames = [str(stimuli_dir / Path(f).name) for f in stimuli_resized_object.filenames]
stimuli_resized = pysaliency.FileStimuli(filenames=reconstructed_filenames)

image_files = stimuli_resized.filenames
random.shuffle(image_files)

for image_path in tqdm(image_files[:NUM_IMAGES_TO_PROCESS]):
    try:
        img_pil = Image.open(image_path).convert("RGB")
        img_np = np.array(img_pil)
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
        mask_stem = Path(image_path).stem
        mask_path = os.path.join(project_root, config.get('mit_all_mask_dir'), f"{mask_stem}.png")
        mask_pil = Image.open(mask_path).convert("L")
        if mask_pil.size != img_pil.size:
            mask_pil = mask_pil.resize(img_pil.size, Image.Resampling.NEAREST)
        mask_np = np.array(mask_pil)
        mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)
        
        embeddings_for_image = extract_segment_embeddings(model, img_tensor, mask_tensor).squeeze(0)
        
        # --- THIS IS THE NEW STYLE NORMALIZATION LOGIC ---
        present_segments = torch.unique(mask_tensor).cpu().numpy()
        valid_embeddings_this_image = []
        metadata_this_image = []
        
        for seg_id in present_segments:
            if seg_id >= model.num_total_segments: continue
            embedding = embeddings_for_image[seg_id].cpu().numpy()
            if np.all(embedding == 0): continue
            
            valid_embeddings_this_image.append(embedding)
            
            segment_pixels = np.where(mask_np == seg_id)
            if segment_pixels[0].size > 0:
                ymin, ymax, xmin, xmax = segment_pixels[0].min(), segment_pixels[0].max(), segment_pixels[1].min(), segment_pixels[1].max()
                segment_img_rgba = Image.fromarray(img_np).convert("RGBA")
                mask_for_alpha = np.zeros_like(mask_np, dtype=np.uint8)
                mask_for_alpha[segment_pixels] = 255
                segment_img_rgba.putalpha(Image.fromarray(mask_for_alpha))
                cropped_segment = segment_img_rgba.crop((xmin, ymin, xmax, ymax))
                metadata_this_image.append({'image_path': image_path, 'segment_id': seg_id, 'segment_image': cropped_segment})

        if valid_embeddings_this_image:
            valid_embeddings_this_image = np.array(valid_embeddings_this_image)
            # Calculate the mean embedding ("style vector") for this image
            image_style_vector = np.mean(valid_embeddings_this_image, axis=0)
            # Subtract the style vector from each segment embedding
            normalized_embeddings = valid_embeddings_this_image - image_style_vector
            
            all_embeddings.extend(normalized_embeddings)
            metadata.extend(metadata_this_image)
        # --- END OF NEW LOGIC ---

    except Exception as e:
        _logger.warning(f"Skipping {os.path.basename(image_path)} due to error: {e}")

# ... (The rest of the UMAP and plotting code is identical and will now work on the normalized embeddings) ...
MIN_SAMPLES_FOR_UMAP = 20
if len(all_embeddings) < MIN_SAMPLES_FOR_UMAP:
    _logger.error(f"Not enough valid segment embeddings extracted ({len(all_embeddings)} found). Need at least {MIN_SAMPLES_FOR_UMAP} for UMAP.")
else:
    all_embeddings = np.array(all_embeddings)
    _logger.info(f"Extracted and normalized {len(all_embeddings)} total segment embeddings.")
    _logger.info("Performing dimensionality reduction with UMAP on normalized embeddings...")
    n_neighbors_for_umap = min(15, len(all_embeddings) - 1)
    reducer = umap.UMAP(n_neighbors=n_neighbors_for_umap, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(all_embeddings)
    _logger.info("UMAP fitting complete.")
    _logger.info("Creating interactive plot. Click on a point to explore neighbors!")
    nn_model = NearestNeighbors(n_neighbors=min(N_NEIGHBORS_TO_SHOW, len(embeddings_2d)), metric='euclidean')
    nn_model.fit(embeddings_2d)
    fig_main, ax_main = plt.subplots(figsize=(10, 8))
    colors = embeddings_2d[:, 1]
    scatter = ax_main.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=5, alpha=0.6, c=colors, cmap='viridis')
    ax_main.set_title("UMAP of Style-Normalized Segment Embeddings\n(Click to Explore)")
    out = widgets.Output()
    def on_click(event):
        if event.inaxes != ax_main: return
        click_coords = np.array([[event.xdata, event.ydata]])
        distances, indices = nn_model.kneighbors(click_coords)
        with out:
            clear_output(wait=True)
            grid_cols, grid_rows = 3, int(np.ceil(N_NEIGHBORS_TO_SHOW / 3))
            fig_neighbors, axes_neighbors = plt.subplots(grid_rows, grid_cols, figsize=(12, 4 * grid_rows))
            if grid_rows == 1: axes_neighbors = np.array([axes_neighbors])
            axes_neighbors = axes_neighbors.flatten()
            fig_neighbors.suptitle(f"Nearest Neighbors in Embedding Space", fontsize=16)
            for i, idx in enumerate(indices[0]):
                meta = metadata[idx]
                ax = axes_neighbors[i]
                ax.imshow(meta['segment_image'])
                ax.set_title(f"{os.path.basename(meta['image_path'])}\nSegment ID: {meta['segment_id']}", fontsize=8)
                ax.axis('off')
            for i in range(len(indices[0]), len(axes_neighbors)):
                axes_neighbors[i].axis('off')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
    fig_main.canvas.mpl_connect('button_press_event', on_click)
    display(out)

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL (v11 - Added Row Titles)
# =================================================================================
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import logging
from types import SimpleNamespace
import pysaliency
from pysaliency.baseline_utils import BaselineModel, CrossvalidatedBaselineModel

# --- 1. Set up Python Path ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src").exists():
        raise FileNotFoundError("Could not find the 'src' directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

# --- 2. Import Your Project's Code ---
from src.datasets.mit1003 import _get_mit_data
from src.data import ImageDataset, FixationMaskTransform

# --- 3. The Visualization Function ---
def show_dataset_samples(num_samples=4):
    """
    Loads SALICON and MIT1003 datasets and displays a grid of sample images
    with their filenames as titles and clear row labels for each dataset.
    """
    print("\n--- Starting Dataset Visualization ---")

    # Mock configuration objects
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass

    cfg = SimpleNamespace(
        paths={"dataset_dir": PROJECT_ROOT / "data" / "pysaliency_datasets"},
        stage=SimpleNamespace(extra={"fold": 0})
    )
    logging.basicConfig(level=logging.INFO, format="[VISUALIZER] %(message)s")
    logger = logging.getLogger("visualize")
    ddp_ctx = MockDDPCtx()

    # Load SALICON Data
    logger.info("Preparing SALICON dataset...")
    salicon_loc = cfg.paths["dataset_dir"] / 'SALICON'
    pysaliency.get_SALICON_train(location=str(salicon_loc.parent))
    salicon_stim, salicon_fix = pysaliency.get_SALICON_train(location=str(salicon_loc.parent))
    salicon_centerbias = BaselineModel(stimuli=salicon_stim, fixations=salicon_fix, bandwidth=0.0217, eps=2e-13)
    salicon_dataset = ImageDataset(salicon_stim, salicon_fix, salicon_centerbias, transform=FixationMaskTransform(sparse=False), average="image")
    logger.info(f"SALICON dataset created with {len(salicon_dataset)} images.")

    # Load MIT1003 Data
    logger.info("Preparing MIT1003 dataset...")
    mit_stim, mit_scan = _get_mit_data(cfg, ddp_ctx, logger)
    mit_stim_train, mit_scan_train = pysaliency.dataset_config.train_split(mit_stim, mit_scan, crossval_folds=10, fold_no=0)
    mit_centerbias = CrossvalidatedBaselineModel(mit_stim, mit_scan, bandwidth=10**-1.667673342543432, eps=10**-14.884189168516073)
    mit_dataset = ImageDataset(mit_stim_train, mit_scan_train, mit_centerbias, transform=FixationMaskTransform(sparse=False), average="image")
    logger.info(f"MIT1003 dataset created with {len(mit_dataset)} images.")

    # Plotting
    logger.info("Generating plot...")
    fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 4, 8.5))
    fig.suptitle("Dataset Image Samples", fontsize=16)

    # --- NEW: Add row titles ---
    axes[0, 0].set_ylabel("SALICON", fontsize=14, weight='bold')
    axes[1, 0].set_ylabel("MIT1003", fontsize=14, weight='bold')

    def prepare_image_for_display(img_chw):
        img_min, img_max = img_chw.min(), img_chw.max()
        img_rescaled = (img_chw - img_min) / (img_max - img_min)
        img_hwc = img_rescaled.transpose(1, 2, 0)
        return img_hwc

    for i in range(num_samples):
        sample_index = i * 10

        # SALICON Sample
        salicon_img_chw = salicon_dataset[sample_index]['image']
        salicon_display_img = prepare_image_for_display(salicon_img_chw)
        salicon_filename = Path(salicon_stim.filenames[sample_index]).name
        
        axes[0, i].imshow(salicon_display_img)
        axes[0, i].set_title(salicon_filename, fontsize=8)
        axes[0, i].axis('off')

        # MIT1003 Sample
        mit_img_chw = mit_dataset[sample_index]['image']
        mit_display_img = prepare_image_for_display(mit_img_chw)
        mit_filename = Path(mit_stim_train.filenames[sample_index]).name
        
        axes[1, i].imshow(mit_display_img)
        axes[1, i].set_title(mit_filename, fontsize=8)
        axes[1, i].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    print("--- Visualization Complete ---")

# --- 4. Run the Visualization ---
show_dataset_samples()

In [ ]:
# =================================================================================
#  CELL 1: PREPARE AND LOAD DATASETS (RUN ONCE)
# =================================================================================
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import logging
from types import SimpleNamespace
import pysaliency
from pysaliency.baseline_utils import BaselineModel, CrossvalidatedBaselineModel

# --- 1. Set up Python Path ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src").exists():
        raise FileNotFoundError("Could not find the 'src' directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

# --- 2. Import Your Project's Code ---
from src.datasets.mit1003 import _get_mit_data
from src.data import ImageDataset, FixationMaskTransform

# --- 3. New Data Preparation Function ---
def prepare_visualization_datasets():
    """
    Loads all required data for visualization and returns it in a dictionary.
    This is the slow part that should only be run once.
    """
    print("\n--- Preparing all visualization datasets. This may take a moment... ---")

    # Mock configuration objects
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass

    cfg = SimpleNamespace(
        paths={"dataset_dir": PROJECT_ROOT / "data" / "pysaliency_datasets"},
        stage=SimpleNamespace(extra={"fold": 0})
    )
    logging.basicConfig(level=logging.INFO, format="[VISUALIZER] %(message)s")
    logger = logging.getLogger("visualize")
    ddp_ctx = MockDDPCtx()

    # --- Load SALICON Data ---
    logger.info("Preparing SALICON dataset...")
    salicon_stim, salicon_fix = pysaliency.get_SALICON_train(location=str(PROJECT_ROOT / "data" / "pysaliency_datasets"))
    salicon_centerbias = BaselineModel(stimuli=salicon_stim, fixations=salicon_fix, bandwidth=0.0217, eps=2e-13)
    salicon_dataset = ImageDataset(salicon_stim, salicon_fix, salicon_centerbias, average="image")

    # --- Load MIT1003 Data ---
    logger.info("Preparing MIT1003 dataset...")
    mit_stim, mit_scan = _get_mit_data(cfg, ddp_ctx, logger)
    mit_stim_train, mit_scan_train = pysaliency.dataset_config.train_split(mit_stim, mit_scan, crossval_folds=10, fold_no=0)
    mit_centerbias = CrossvalidatedBaselineModel(mit_stim, mit_scan, bandwidth=10**-1.667673342543432, eps=10**-14.884189168516073)
    mit_dataset = ImageDataset(mit_stim_train, mit_scan_train, mit_centerbias, average="image")

    # --- Package everything for return ---
    datasets = {
        'salicon': {
            'stim': salicon_stim,
            'fix': salicon_fix,
            'dataset': salicon_dataset
        },
        'mit': {
            'stim': mit_stim_train,
            'fix': mit_scan_train,
            'dataset': mit_dataset
        }
    }
    
    print("\n✅ All datasets are loaded and ready for plotting.")
    return datasets

# --- 4. Execute the Preparation Function ---
# This line runs the function and stores the result in a global variable
viz_datasets = prepare_visualization_datasets()

In [ ]:
# =================================================================================
#  CELL 2: PLOTTING FUNCTION (RUN REPEATEDLY)
# =================================================================================
def plot_fixation_overlays(datasets, salicon_index=22, mit_index=26, num_subjects=5):
    """
    Takes pre-loaded data and generates a plot with scanpath overlays.
    You can call this function multiple times with different indices.
    """
    print(f"\n--- Generating plot for SALICON index {salicon_index} and MIT1003 index {mit_index} ---")

    # --- Unpack pre-loaded data ---
    salicon_stim = datasets['salicon']['stim']
    salicon_fix = datasets['salicon']['fix']
    salicon_dataset = datasets['salicon']['dataset']
    
    mit_stim_train = datasets['mit']['stim']
    mit_scan_train = datasets['mit']['fix']
    mit_dataset = datasets['mit']['dataset']
    
    # --- Plotting ---
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(f"Scanpath Overlays from {num_subjects} Sample Subjects", fontsize=16)

    def prepare_image_for_display(img_chw):
        img_min, img_max = img_chw.min(), img_chw.max()
        img_rescaled = (img_chw - img_min) / (img_max - img_min)
        return img_rescaled.transpose(1, 2, 0)

    # --- 1. Process and Plot SALICON Image ---
    ax_salicon = axes[0]
    salicon_img_chw = salicon_dataset[salicon_index]['image']
    salicon_display_img = prepare_image_for_display(salicon_img_chw)
    ax_salicon.imshow(salicon_display_img)
    
    fixations_for_image = salicon_fix[salicon_fix.n == salicon_index]
    subjects_for_image = np.unique(fixations_for_image.subject)
    subjects_to_plot = subjects_for_image[:num_subjects]
    
    cmap = plt.get_cmap('gist_rainbow', len(subjects_to_plot))
    for i, subject_id in enumerate(subjects_to_plot):
        subject_fixations = fixations_for_image[fixations_for_image.subject == subject_id]
        sort_indices = np.argsort(subject_fixations.t)
        sorted_subject_fixations = subject_fixations[sort_indices]
        ax_salicon.plot(sorted_subject_fixations.x, sorted_subject_fixations.y, 
                        marker='o', markersize=8, linestyle='-', linewidth=2,
                        color=cmap(i), alpha=0.7, label=f'Subj. {i+1}')

    salicon_filename = Path(salicon_stim.filenames[salicon_index]).name
    ax_salicon.set_title(f"SALICON: {salicon_filename}", fontsize=12)
    ax_salicon.axis('off')
    ax_salicon.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title="Subjects")

    # --- 2. Process and Plot MIT1003 Image ---
    ax_mit = axes[1]
    mit_img_chw = mit_dataset[mit_index]['image']
    mit_display_img = prepare_image_for_display(mit_img_chw)

    scanpaths_for_image_mit = mit_scan_train.scanpaths[mit_scan_train.scanpaths.n == mit_index]
    list_of_x_paths = scanpaths_for_image_mit.xs[:num_subjects]
    list_of_y_paths = scanpaths_for_image_mit.ys[:num_subjects]
    
    ax_mit.imshow(mit_display_img)
    cmap = plt.get_cmap('viridis', len(list_of_x_paths))
    for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
        ax_mit.plot(x_path, y_path, 
                    marker='o', markersize=8, linestyle='-', linewidth=2,
                    color=cmap(i), alpha=0.7, label=f'Subj. {i+1}')

    mit_filename = Path(mit_stim_train.filenames[mit_index]).name
    ax_mit.set_title(f"MIT1003: {mit_filename}", fontsize=12)
    ax_mit.axis('off')
    ax_mit.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title="Subjects")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    print("--- Plot Generation Complete ---")

# --- 4. Example Usage ---
# Now you can call the plotting function with the loaded data.
# Run this line to see the default plot:
plot_fixation_overlays(viz_datasets)

# Run this line to see a plot with different images:
# plot_fixation_overlays(viz_datasets, salicon_index=500, mit_index=100)

# Run this line to see a plot with 10 subjects instead of 5:
plot_fixation_overlays(viz_datasets, num_subjects=10)

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL for DinoGaze-SPADE (Publication Quality v1.4 - GridSpec Fix)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")
from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/dinogaze_spade_v1_sam64_fold0/step-0005.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14', 'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1], 'dino_semantic_feature_layer_idx': -1,
    'num_total_sam_segments': 64, 'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True, 'fold': 0,
}

# --- General Settings ---
RANDOM_SEED = 123
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. MANUALLY BUILD THE CONFIG OBJECT ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"}, stage=SimpleNamespace(model_key=MODEL_KEY, dataset_key=DATASET_KEY, extra={"mit_all_mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}))

# --- 4. HELPER FUNCTIONS ---
def load_data_for_visualization(cfg, random_seed):
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data"); ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data
    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0); val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)
    random.seed(random_seed); image_path = random.choice(val_stim.filenames)
    print(f"Randomly selected image: {Path(image_path).name}")
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mit_all_mask_dir"); mask_path = mask_dir / f"{Path(image_path).stem}.png"
    stimulus_index = stimuli_resized.filenames.index(str(image_path)); scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    return {"image_path": image_path, "mask_path": mask_path, "scanpaths_x": scanpaths_for_image.xs, "scanpaths_y": scanpaths_for_image.ys}

def load_model_for_visualization(cfg, checkpoint_path, device):
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False); model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    model.load_state_dict(cleaned_state_dict, strict=False)
    model.eval()
    return model

# --- 5. EXECUTE LOADING AND VISUALIZATION ---
viz_data = load_data_for_visualization(cfg, RANDOM_SEED)
model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)
original_image = Image.open(viz_data['image_path']).convert('RGB')
preprocess = T.Compose([T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
image_tensor = preprocess(original_image).unsqueeze(0).to(DEVICE)
mask_image = Image.open(viz_data['mask_path']).convert('L'); mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)
list_of_x_paths, list_of_y_paths = viz_data['scanpaths_x'], viz_data['scanpaths_y']

# --- FIX: Restore the hook logic ---
captured_maps = {}
def get_spade_hook(name):
    def hook(module, input, output): captured_maps[name] = output.detach().squeeze(0).mean(dim=0).cpu().numpy()
    return hook
hook_handles = []
if hasattr(model, 'saliency_network') and hasattr(model.saliency_network, 'spade_ln0'):
    spade_layers_to_hook = { 'SPADE_0': model.saliency_network.spade_ln0, 'SPADE_1': model.saliency_network.spade_ln1, 'SPADE_2': model.saliency_network.spade_ln2 }
    for name, layer in spade_layers_to_hook.items():
        hook_handles.extend([layer.mlp_gamma.register_forward_hook(get_spade_hook(f'{name}_Gamma')), layer.mlp_beta.register_forward_hook(get_spade_hook(f'{name}_Beta'))])
    print(f"✅ Attached {len(hook_handles)} hooks to SPADE layers.")

with torch.no_grad():
    # Full model pass will trigger hooks
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
    log_density_map = model(image_tensor, centerbias, segmentation_mask=mask_tensor)
    prob_map = torch.exp(log_density_map)
    
    # Manual pass for sharp map
    extracted_feature_maps = model.features(image_tensor)
    readout_h = math.ceil(image_tensor.shape[2] / model.readout_factor); readout_w = math.ceil(image_tensor.shape[3] / model.readout_factor)
    processed_features_list = [F.interpolate(f, size=(readout_h, readout_w), mode='bilinear') for f in extracted_feature_maps]
    concatenated_backbone_features = torch.cat(processed_features_list, dim=1)
    semantic_feature_map = extracted_feature_maps[model.semantic_feature_layer_idx]
    S_painted_map_full_res = model._create_painted_semantic_map_vectorized(semantic_feature_map, mask_tensor)
    saliency_head_output = model.saliency_network(concatenated_backbone_features, S_painted_map_full_res)
    saliency_grid_sharp = model.fixation_selection_network((saliency_head_output, None))
    saliency_map_tensor_sharp = F.interpolate(saliency_grid_sharp, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)

saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()
saliency_map_np_log_density = log_density_map.squeeze().cpu().numpy()
saliency_map_np_prob = prob_map.squeeze().cpu().numpy()

# --- FIX: Remove hooks after all forward passes are complete ---
for handle in hook_handles: handle.remove()
if hook_handles: print("Hooks removed.")

def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map
heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)
heatmap_log_density = normalize_for_viz(saliency_map_np_log_density)
heatmap_prob = normalize_for_viz(saliency_map_np_prob)
for key in captured_maps: captured_maps[key] = normalize_for_viz(captured_maps[key])

# ================================================================================
# --- PLOT 1: MAIN ANALYSIS (PUBLICATION QUALITY) ---
# ================================================================================
print(f"\nCreating Publication-Quality Main Analysis visualization...")
FONTSIZE = 16; CMAP_VIZ = 'viridis'; DPI = 300

fig = plt.figure(figsize=(18, 12), dpi=DPI)
gs = gridspec.GridSpec(2, 4, figure=fig, width_ratios=[1, 1, 1, 0.05])
fig.suptitle(f'Saliency Analysis: {cfg.stage.model_key}', fontsize=FONTSIZE + 4)
ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1]); ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0]); ax_e = fig.add_subplot(gs[1, 1]); ax_f = fig.add_subplot(gs[1, 2])
cax = fig.add_subplot(gs[:, 3])

ax_a.imshow(original_image); ax_a.set_title('(a) Original Image', fontsize=FONTSIZE)
ax_b.imshow(mask_np, cmap='nipy_spectral'); ax_b.set_title('(b) Segmentation Mask', fontsize=FONTSIZE)
ax_c.imshow(original_image)
cmap_gt = plt.get_cmap('viridis', len(list_of_x_paths))
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    ax_c.plot(x_path, y_path, marker='o', lw=1.5, ms=4, alpha=0.8, color=cmap_gt(i))
ax_c.set_title('(c) Ground-Truth Scanpaths', fontsize=FONTSIZE)
ax_d.imshow(heatmap_sharp, cmap=CMAP_VIZ, vmin=0, vmax=1); ax_d.set_title('(d) Sharp Heatmap', fontsize=FONTSIZE)
ax_e.imshow(heatmap_log_density, cmap=CMAP_VIZ, vmin=0, vmax=1); ax_e.set_title('(e) Log-Density Heatmap', fontsize=FONTSIZE)
im_prob = ax_f.imshow(heatmap_prob, cmap=CMAP_VIZ, vmin=0, vmax=1); ax_f.set_title('(f) Probability Heatmap', fontsize=FONTSIZE)
for ax in [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f]: ax.axis('off')
fig.colorbar(im_prob, cax=cax, label='Normalized Saliency')
fig.tight_layout(rect=[0, 0, 1, 0.96])

output_path_main = PROJECT_ROOT / f'output/{cfg.stage.model_key}_main_analysis_paper.png'
plt.savefig(output_path_main, bbox_inches='tight'); plt.show(); plt.close(fig)
print(f"✅ Publication-quality analysis saved to: {output_path_main}")

# ================================================================================
# --- PLOT 2: SPADE MODULATION MAPS (PUBLICATION QUALITY) ---
# ================================================================================
if captured_maps:
    print(f"Creating Publication-Quality SPADE Maps visualization...")
    fig_spade = plt.figure(figsize=(18, 12), dpi=DPI)
    gs_spade = gridspec.GridSpec(2, 4, figure=fig_spade, width_ratios=[1, 1, 1, 0.05])
    fig_spade.suptitle('SPADE Modulation Analysis', fontsize=FONTSIZE + 4)
    ax_g0 = fig_spade.add_subplot(gs_spade[0, 0]); ax_g1 = fig_spade.add_subplot(gs_spade[0, 1]); ax_g2 = fig_spade.add_subplot(gs_spade[0, 2])
    ax_b0 = fig_spade.add_subplot(gs_spade[1, 0]); ax_b1 = fig_spade.add_subplot(gs_spade[1, 1]); ax_b2 = fig_spade.add_subplot(gs_spade[1, 2])
    cax_g = fig_spade.add_subplot(gs_spade[0, 3]); cax_b = fig_spade.add_subplot(gs_spade[1, 3])
    
    im_g2 = ax_g2.imshow(captured_maps['SPADE_2_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1)
    ax_g0.imshow(captured_maps['SPADE_0_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_g0.set_title('(a) Gamma Map (SPADE 0)', fontsize=FONTSIZE)
    ax_g1.imshow(captured_maps['SPADE_1_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_g1.set_title('(b) Gamma Map (SPADE 1)', fontsize=FONTSIZE)
    ax_g2.set_title('(c) Gamma Map (SPADE 2)', fontsize=FONTSIZE)
    ax_g0.set_ylabel('Gamma (Scaling)', fontsize=FONTSIZE, weight='bold')
    
    im_b2 = ax_b2.imshow(captured_maps['SPADE_2_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1)
    ax_b0.imshow(captured_maps['SPADE_0_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_b0.set_title('(d) Beta Map (SPADE 0)', fontsize=FONTSIZE)
    ax_b1.imshow(captured_maps['SPADE_1_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_b1.set_title('(e) Beta Map (SPADE 1)', fontsize=FONTSIZE)
    ax_b2.set_title('(f) Beta Map (SPADE 2)', fontsize=FONTSIZE)
    ax_b0.set_ylabel('Beta (Shifting)', fontsize=FONTSIZE, weight='bold')
    
    for ax in [ax_g0, ax_g1, ax_g2, ax_b0, ax_b1, ax_b2]: ax.axis('off')
    fig_spade.colorbar(im_g2, cax=cax_g, label='Normalized Gamma Value')
    fig_spade.colorbar(im_b2, cax=cax_b, label='Normalized Beta Value')
    fig_spade.tight_layout(rect=[0, 0, 1, 0.96])
    
    output_path_spade = PROJECT_ROOT / f'output/{cfg.stage.model_key}_spade_maps_paper.png'
    plt.savefig(output_path_spade, bbox_inches='tight'); plt.show(); plt.close(fig_spade)
    print(f"✅ Publication-quality SPADE maps saved to: {output_path_spade}")

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL for DeepGaze III (Publication Quality)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/deepgaze3_fold0/final_best_val.pth'

# --- Registry Keys ---
MODEL_KEY = 'deepgaze3'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'downsample': 1.0,\
    'readout_factor': 4,
    'initial_sigma': 8.0,
    'fold': 0,
}

# --- General Settings ---
RANDOM_SEED = 123
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. MANUALLY BUILD THE CONFIG OBJECT (NO YAML NEEDED) ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(
    paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
    stage=SimpleNamespace(
        model_key=MODEL_KEY,
        dataset_key=DATASET_KEY,
        extra=MODEL_PARAMS
    )
)
print("✅ Config object created.")

# --- 4. HELPER FUNCTIONS ---
def load_data_for_visualization(cfg, random_seed):
    print("\n--- Loading data using registry logic ---")
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data")
    ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data

    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0)
    val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)
    
    random.seed(random_seed)
    image_path = random.choice(val_stim.filenames)
    print(f"Randomly selected image: {Path(image_path).name}")

    stimulus_index = stimuli_resized.filenames.index(str(image_path))
    scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    
    return {
        "image_path": image_path,
        "scanpaths_x": scanpaths_for_image.xs,
        "scanpaths_y": scanpaths_for_image.ys,
    }

def load_model_for_visualization(cfg, checkpoint_path, device):
    print("\n--- Building model from registry ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    print(f"--- Loading checkpoint: {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    missing_keys, unexpected_keys = model.load_state_dict(cleaned_state_dict, strict=False)
    print(f"Loaded weights. Missing keys: {missing_keys}, Unexpected keys: {unexpected_keys}")
    model.eval()
    print("✅ Model built and weights loaded successfully.")
    return model

# --- 5. EXECUTE LOADING AND VISUALIZATION ---
viz_data = load_data_for_visualization(cfg, RANDOM_SEED)
model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)

# --- Prepare Inputs ---
print("\n--- Preparing image for model ---")
original_image = Image.open(viz_data['image_path']).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
list_of_x_paths, list_of_y_paths = viz_data['scanpaths_x'], viz_data['scanpaths_y']

# --- Generate Saliency Maps ---
with torch.no_grad():
    print("Generating all saliency maps...")
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
    log_density_map = model(image_tensor, centerbias)
    saliency_map_np_log_density = log_density_map.squeeze().cpu().numpy()
    prob_map = torch.exp(log_density_map)
    saliency_map_np_prob = prob_map.squeeze().cpu().numpy()
    x = F.interpolate(image_tensor, scale_factor=1 / model.downsample)
    features_list = model.features(x)
    readout_shape = [math.ceil(image_tensor.shape[2] / model.downsample / model.readout_factor), 
                     math.ceil(image_tensor.shape[3] / model.downsample / model.readout_factor)]
    x_resized = [F.interpolate(item, readout_shape) for item in features_list]
    concatenated_features = torch.cat(x_resized, dim=1)
    saliency_grid = model.saliency_network(concatenated_features)
    score_grid_before_finalizer = model.fixation_selection_network((saliency_grid, None))
    saliency_map_tensor_sharp = F.interpolate(score_grid_before_finalizer, size=image_tensor.shape[2:], mode='bilinear')
    saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()

# --- Normalize all maps for visualization ---
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map

heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)
heatmap_log_density = normalize_for_viz(saliency_map_np_log_density)
heatmap_prob = normalize_for_viz(saliency_map_np_prob)

# ================================================================================
# --- PLOT 1: MAIN ANALYSIS (PUBLICATION QUALITY) ---
# ================================================================================
print(f"Creating Publication-Quality Main Analysis visualization...")

# --- Paper-Quality Settings ---
FONTSIZE = 16
CMAP_VIZ = 'viridis'
DPI = 300 # Standard for publications

fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=DPI, constrained_layout=True)
fig.suptitle(f'Saliency Analysis: {cfg.stage.model_key}', fontsize=FONTSIZE + 4)

# --- Row 1: Inputs & Sharp Map ---
axes[0, 0].imshow(original_image)
axes[0, 0].set_title('(a) Original Image', fontsize=FONTSIZE)
axes[0, 1].imshow(original_image)
cmap_gt = plt.get_cmap('viridis', len(list_of_x_paths))
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    axes[0, 1].plot(x_path, y_path, marker='o', lw=1.5, ms=4, alpha=0.8, color=cmap_gt(i))
axes[0, 1].set_title('(b) Ground-Truth Scanpaths', fontsize=FONTSIZE)
im_sharp = axes[0, 2].imshow(heatmap_sharp, cmap=CMAP_VIZ, vmin=0, vmax=1)
axes[0, 2].set_title('(c) Sharp Heatmap', fontsize=FONTSIZE)

# --- Row 2: Model Final Outputs ---
axes[1, 0].imshow(heatmap_log_density, cmap=CMAP_VIZ, vmin=0, vmax=1)
axes[1, 0].set_title('(d) Log-Density Heatmap', fontsize=FONTSIZE)
im_prob = axes[1, 1].imshow(heatmap_prob, cmap=CMAP_VIZ, vmin=0, vmax=1)
axes[1, 1].set_title('(e) Probability Heatmap', fontsize=FONTSIZE)
axes[1, 2].axis('off') # Keep this panel empty for balance and colorbar space

for ax in axes.flat:
    ax.axis('off')

# --- Add a shared colorbar for all saliency maps ---
fig.colorbar(im_prob, ax=axes[:, 2], shrink=0.6, pad=0.02, label='Normalized Saliency')

output_path_main = PROJECT_ROOT / f'output/{cfg.stage.model_key}_main_analysis_paper.png'
output_path_main.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path_main, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"✅ Publication-quality analysis saved to: {output_path_main}")

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL for DeepgazeSpadeV3 (Publication Quality v1.4 - GridSpec Fix)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
# --- FIX: Import GridSpec for perfect layout control ---
import matplotlib.gridspec as gridspec
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/deepgaze_spade_v3_sam64_fold0/step-0004.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'deepgaze_spade_v3'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_semantic_layer_idx': -1,
    'num_total_segments': 64,
    'downsample': 1.0,
    'readout_factor': 4,
    'saliency_map_factor': 4,
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'fold': 0,
}

# --- General Settings ---
RANDOM_SEED = 123
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- (Setup code is unchanged, snipped for brevity) ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"}, stage=SimpleNamespace(model_key=MODEL_KEY, dataset_key=DATASET_KEY, extra={"mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}))
def load_data_for_visualization(cfg, random_seed):
    class MockDDPCtx:
        is_master = True
        enabled = False
        rank = 0
        world = 1
        device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data")
    ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data
    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0); val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)
    random.seed(random_seed); image_path = random.choice(val_stim.filenames)
    print(f"Randomly selected image: {Path(image_path).name}")
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mask_dir"); mask_path = mask_dir / f"{Path(image_path).stem}.png"
    if not mask_path.exists(): raise FileNotFoundError(f"Could not find mask at: {mask_path}")
    stimulus_index = stimuli_resized.filenames.index(str(image_path)); scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    return {"image_path": image_path, "mask_path": mask_path, "scanpaths_x": scanpaths_for_image.xs, "scanpaths_y": scanpaths_for_image.ys}
def load_model_for_visualization(cfg, checkpoint_path, device):
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False); model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    model.load_state_dict(cleaned_state_dict, strict=False)
    model.eval()
    return model
viz_data = load_data_for_visualization(cfg, RANDOM_SEED)
model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)
original_image = Image.open(viz_data['image_path']).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
mask_image = Image.open(viz_data['mask_path']).convert('L'); mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)
list_of_x_paths, list_of_y_paths = viz_data['scanpaths_x'], viz_data['scanpaths_y']
captured_maps = {}
def get_spade_hook(name):
    def hook(module, input, output): captured_maps[name] = output.detach().squeeze(0).mean(dim=0).cpu().numpy()
    return hook
hook_handles = []
if hasattr(model, 'saliency_network') and hasattr(model.saliency_network, 'spade_ln0'):
    spade_layers_to_hook = { 'SPADE_0': model.saliency_network.spade_ln0, 'SPADE_1': model.saliency_network.spade_ln1, 'SPADE_2': model.saliency_network.spade_ln2, }
    for name, layer in spade_layers_to_hook.items():
        hook_handles.extend([layer.mlp_gamma.register_forward_hook(get_spade_hook(f'{name}_Gamma')), layer.mlp_beta.register_forward_hook(get_spade_hook(f'{name}_Beta'))])
with torch.no_grad():
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
    log_density_map = model(image_tensor, centerbias, segmentation_mask=mask_tensor)
    prob_map = torch.exp(log_density_map)
    orig_shape_hw = image_tensor.shape[2:]
    dino_feature_maps = model.dino_features(image_tensor)
    semantic_dino_patches = dino_feature_maps[-1]
    densenet_feature_maps = model.densenet_features(image_tensor)
    painted_map = model._create_painted_semantic_map_vectorized(semantic_dino_patches, mask_tensor)
    readout_shape = (math.ceil(orig_shape_hw[0] / model.readout_factor), math.ceil(orig_shape_hw[1] / model.readout_factor))
    processed_features_list = [F.interpolate(f, size=readout_shape, mode='bilinear') for f in densenet_feature_maps]
    concatenated_features = torch.cat(processed_features_list, dim=1)
    saliency_output = model.saliency_network(concatenated_features, painted_map)
    score_grid_before_finalizer = model.fixation_selection_network((saliency_output, None))
    saliency_map_tensor_sharp = F.interpolate(score_grid_before_finalizer, size=orig_shape_hw, mode='bilinear')
saliency_map_np_sharp = saliency_map_tensor_sharp.squeeze().cpu().numpy()
saliency_map_np_log_density = log_density_map.squeeze().cpu().numpy()
saliency_map_np_prob = prob_map.squeeze().cpu().numpy()
for handle in hook_handles: handle.remove()
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map
heatmap_sharp = normalize_for_viz(saliency_map_np_sharp)
heatmap_log_density = normalize_for_viz(saliency_map_np_log_density)
heatmap_prob = normalize_for_viz(saliency_map_np_prob)
for key in captured_maps: captured_maps[key] = normalize_for_viz(captured_maps[key])
# ...

# ================================================================================
# --- PLOT 1: MAIN ANALYSIS (PUBLICATION QUALITY) ---
# ================================================================================
print(f"\nCreating Publication-Quality Main Analysis visualization...")

FONTSIZE = 16
CMAP_VIZ = 'viridis'
DPI = 300

# --- FIX: Use GridSpec to create a dedicated column for the colorbar ---
fig = plt.figure(figsize=(18, 12), dpi=DPI)
gs = gridspec.GridSpec(2, 4, figure=fig, width_ratios=[1, 1, 1, 0.05])
fig.suptitle(f'Saliency Analysis: {cfg.stage.model_key}', fontsize=FONTSIZE + 4)

# Create axes using the gridspec
ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1]); ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0]); ax_e = fig.add_subplot(gs[1, 1]); ax_f = fig.add_subplot(gs[1, 2])
cax = fig.add_subplot(gs[:, 3]) # Colorbar axis spans all rows

# --- Row 1: Inputs ---
ax_a.imshow(original_image)
ax_a.set_title('(a) Original Image', fontsize=FONTSIZE)
ax_b.imshow(mask_np, cmap='nipy_spectral')
ax_b.set_title('(b) Segmentation Mask', fontsize=FONTSIZE)
ax_c.imshow(original_image)
cmap_gt = plt.get_cmap('viridis', len(list_of_x_paths))
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    # --- FIX: Remove label and legend for a cleaner paper figure ---
    ax_c.plot(x_path, y_path, marker='o', lw=1.5, ms=4, alpha=0.8, color=cmap_gt(i))
ax_c.set_title('(c) Ground-Truth Scanpaths', fontsize=FONTSIZE)

# --- Row 2: Model Outputs ---
ax_d.imshow(heatmap_sharp, cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_d.set_title('(d) Sharp Heatmap', fontsize=FONTSIZE)
ax_e.imshow(heatmap_log_density, cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_e.set_title('(e) Log-Density Heatmap', fontsize=FONTSIZE)
im_prob = ax_f.imshow(heatmap_prob, cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_f.set_title('(f) Probability Heatmap', fontsize=FONTSIZE)

for ax in [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f]:
    ax.axis('off')

fig.colorbar(im_prob, cax=cax, label='Normalized Saliency')

# Let GridSpec handle the layout automatically
fig.tight_layout(rect=[0, 0, 1, 0.96])

output_path_main = PROJECT_ROOT / f'output/{cfg.stage.model_key}_main_analysis_paper.png'
output_path_main.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path_main, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"✅ Publication-quality analysis saved to: {output_path_main}")

# ================================================================================
# --- PLOT 2: SPADE MODULATION MAPS (PUBLICATION QUALITY) ---
# ================================================================================
if captured_maps:
    print(f"Creating Publication-Quality SPADE Maps visualization...")
    # --- FIX: Use GridSpec for the SPADE plot as well for consistency ---
    fig_spade = plt.figure(figsize=(18, 12), dpi=DPI)
    gs_spade = gridspec.GridSpec(2, 4, figure=fig_spade, width_ratios=[1, 1, 1, 0.05])
    fig_spade.suptitle('SPADE Modulation Analysis', fontsize=FONTSIZE + 4)
    
    ax_g0 = fig_spade.add_subplot(gs_spade[0, 0]); ax_g1 = fig_spade.add_subplot(gs_spade[0, 1]); ax_g2 = fig_spade.add_subplot(gs_spade[0, 2])
    ax_b0 = fig_spade.add_subplot(gs_spade[1, 0]); ax_b1 = fig_spade.add_subplot(gs_spade[1, 1]); ax_b2 = fig_spade.add_subplot(gs_spade[1, 2])
    cax_g = fig_spade.add_subplot(gs_spade[0, 3]) # Colorbar for Gamma row
    cax_b = fig_spade.add_subplot(gs_spade[1, 3]) # Colorbar for Beta row

    im_g2 = ax_g2.imshow(captured_maps['SPADE_2_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1)
    ax_g0.imshow(captured_maps['SPADE_0_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_g0.set_title('(a) Gamma Map (SPADE 0)', fontsize=FONTSIZE)
    ax_g1.imshow(captured_maps['SPADE_1_Gamma'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_g1.set_title('(b) Gamma Map (SPADE 1)', fontsize=FONTSIZE)
    ax_g2.set_title('(c) Gamma Map (SPADE 2)', fontsize=FONTSIZE)
    ax_g0.set_ylabel('Gamma (Scaling)', fontsize=FONTSIZE, weight='bold')

    im_b2 = ax_b2.imshow(captured_maps['SPADE_2_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1)
    ax_b0.imshow(captured_maps['SPADE_0_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_b0.set_title('(d) Beta Map (SPADE 0)', fontsize=FONTSIZE)
    ax_b1.imshow(captured_maps['SPADE_1_Beta'], cmap=CMAP_VIZ, vmin=0, vmax=1); ax_b1.set_title('(e) Beta Map (SPADE 1)', fontsize=FONTSIZE)
    ax_b2.set_title('(f) Beta Map (SPADE 2)', fontsize=FONTSIZE)
    ax_b0.set_ylabel('Beta (Shifting)', fontsize=FONTSIZE, weight='bold')

    for ax in [ax_g0, ax_g1, ax_g2, ax_b0, ax_b1, ax_b2]:
        ax.axis('off')
    
    fig_spade.colorbar(im_g2, cax=cax_g, label='Normalized Gamma Value')
    fig_spade.colorbar(im_b2, cax=cax_b, label='Normalized Beta Value')

    fig_spade.tight_layout(rect=[0, 0, 1, 0.96])
    
    output_path_spade = PROJECT_ROOT / f'output/{cfg.stage.model_key}_spade_maps_paper.png'
    plt.savefig(output_path_spade, bbox_inches='tight')
    plt.show()
    plt.close(fig_spade)
    print(f"✅ Publication-quality SPADE maps saved to: {output_path_spade}")

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL: DeepGaze III vs. DeepGaze SPADE V3 (Publication Quality v1.4)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
# --- FIX: Import GridSpec for perfect layout control ---
import matplotlib.gridspec as gridspec
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- Model 1: Standard DeepGaze III ---
DG3_MODEL_KEY = 'deepgaze3'
DG3_CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_scanpath_frozen/deepgaze3_fold0/final_best_val.pth'
DG3_MODEL_PARAMS = { 'downsample': 1.0, 'readout_factor': 4, 'initial_sigma': 8.0 }

# --- Model 2: DeepGaze SPADE V3 (Hybrid) ---
SPADE_V3_MODEL_KEY = 'deepgaze_spade_v3'
SPADE_V3_CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/deepgaze_spade_v3_sam64_fold0/step-0004.pth'
SPADE_V3_MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14', 'dino_semantic_layer_idx': -1,
    'num_total_segments': 64, 'downsample': 1.0, 'readout_factor': 4,
    'saliency_map_factor': 4, 'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
}

# --- Shared Data & General Settings ---
DATASET_KEY = 'MIT1003'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'
RANDOM_SEED = 43
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- (Setup code is unchanged, snipped for brevity) ---
print("\n--- Manually building config objects ---")
cfg_dg3 = SimpleNamespace(paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"}, stage=SimpleNamespace(model_key=DG3_MODEL_KEY, dataset_key=DATASET_KEY, extra={'fold': 0, **DG3_MODEL_PARAMS}))
cfg_spade = SimpleNamespace(paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"}, stage=SimpleNamespace(model_key=SPADE_V3_MODEL_KEY, dataset_key=DATASET_KEY, extra={'mask_dir': MIT_MASK_DIR, 'fold': 0, **SPADE_V3_MODEL_PARAMS}))
print("✅ Config objects created.")
def load_data_for_visualization(cfg, random_seed):
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data"); ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data
    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0); val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)
    random.seed(random_seed); image_path = random.choice(val_stim.filenames)
    print(f"Randomly selected image: {Path(image_path).name}")
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mask_dir"); mask_path = mask_dir / f"{Path(image_path).stem}.png"
    if not mask_path.exists(): raise FileNotFoundError(f"Could not find mask at: {mask_path}")
    stimulus_index = stimuli_resized.filenames.index(str(image_path)); scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    return {"image_path": image_path, "mask_path": mask_path, "scanpaths_x": scanpaths_for_image.xs, "scanpaths_y": scanpaths_for_image.ys}
def load_model_for_visualization(cfg, checkpoint_path, device):
    print(f"\n--- Building model '{cfg.stage.model_key}' from registry ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False); model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    model.load_state_dict(cleaned_state_dict, strict=False)
    model.eval()
    return model
viz_data = load_data_for_visualization(cfg_spade, RANDOM_SEED)
model_dg3 = load_model_for_visualization(cfg_dg3, DG3_CHECKPOINT_PATH, DEVICE)
model_spade = load_model_for_visualization(cfg_spade, SPADE_V3_CHECKPOINT_PATH, DEVICE)
print("\n--- Preparing images and masks for models ---")
original_image = Image.open(viz_data['image_path']).convert('RGB')
image_np = np.array(original_image)
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
mask_image = Image.open(viz_data['mask_path']).convert('L'); mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)
list_of_x_paths, list_of_y_paths = viz_data['scanpaths_x'], viz_data['scanpaths_y']
with torch.no_grad():
    print("Generating maps for both models...")
    centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
    log_density_dg3 = model_dg3(image_tensor, centerbias); prob_map_dg3 = torch.exp(log_density_dg3)
    log_density_spade = model_spade(image_tensor, centerbias, segmentation_mask=mask_tensor); prob_map_spade = torch.exp(log_density_spade)
maps = {
    'dg3_log': log_density_dg3.squeeze().cpu().numpy(), 'dg3_prob': prob_map_dg3.squeeze().cpu().numpy(),
    'spade_log': log_density_spade.squeeze().cpu().numpy(), 'spade_prob': prob_map_spade.squeeze().cpu().numpy(),
}
def normalize_for_viz(np_map):
    min_val, max_val = np_map.min(), np_map.max()
    return (np_map - min_val) / (max_val - min_val) if max_val > min_val else np_map
heatmaps = {key: normalize_for_viz(val) for key, val in maps.items()}
# ...

# ================================================================================
# --- 8. PLOT: MAIN COMPARISON (PUBLICATION QUALITY) ---
# ================================================================================
print("\nCreating Publication-Quality Comparison visualization...")
FONTSIZE = 16
CMAP_VIZ = 'viridis'
DPI = 300

# --- FIX: Use GridSpec to create a dedicated column for the colorbar ---
fig = plt.figure(figsize=(18, 18), dpi=DPI)
gs = gridspec.GridSpec(3, 4, figure=fig, width_ratios=[1, 1, 1, 0.05])
fig.suptitle('Model Comparison: DeepGaze III vs. DeepGaze-SPADE', fontsize=FONTSIZE + 4)

# Create axes using the gridspec
ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1]); ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0]); ax_e = fig.add_subplot(gs[1, 1]); ax_f = fig.add_subplot(gs[1, 2])
ax_g = fig.add_subplot(gs[2, 0]); ax_h = fig.add_subplot(gs[2, 1]); ax_i = fig.add_subplot(gs[2, 2])
cax_dg3 = fig.add_subplot(gs[1, 3])   # Colorbar axis for row 1
cax_spade = fig.add_subplot(gs[2, 3]) # Colorbar axis for row 2

# --- Row 1: Common Inputs ---
ax_a.imshow(original_image)
ax_a.set_title('(a) Original Image', fontsize=FONTSIZE)
ax_b.imshow(mask_np, cmap='nipy_spectral')
ax_b.set_title('(b) Segmentation Mask', fontsize=FONTSIZE)
ax_c.imshow(original_image)
cmap_gt = plt.get_cmap('viridis', len(list_of_x_paths))
for i, (x_path, y_path) in enumerate(zip(list_of_x_paths, list_of_y_paths)):
    # --- FIX: Remove the label and legend call for a cleaner look ---
    ax_c.plot(x_path, y_path, marker='o', lw=1.5, ms=4, alpha=0.8, color=cmap_gt(i))
ax_c.set_title('(c) Ground-Truth Scanpaths', fontsize=FONTSIZE)

# --- Row 2: DeepGaze III Outputs ---
ax_d.set_ylabel('DeepGaze III', fontsize=FONTSIZE, weight='bold')
ax_d.imshow(heatmaps['dg3_log'], cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_d.set_title('(d) Log-Density Heatmap', fontsize=FONTSIZE)
im_dg3 = ax_e.imshow(heatmaps['dg3_prob'], cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_e.set_title('(e) Probability Heatmap', fontsize=FONTSIZE)
ax_f.imshow(original_image)
ax_f.imshow(heatmaps['dg3_log'], cmap=CMAP_VIZ, vmin=0, vmax=1, alpha=0.5)
ax_f.set_title('(f) Log-density Overlay', fontsize=FONTSIZE)
fig.colorbar(im_dg3, cax=cax_dg3, label='Normalized Saliency (DG3)')

# --- Row 3: DeepGaze-SPADE Outputs ---
ax_g.set_ylabel('DeepGaze-SPADE', fontsize=FONTSIZE, weight='bold')
ax_g.imshow(heatmaps['spade_log'], cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_g.set_title('(g) Log-Density Heatmap', fontsize=FONTSIZE)
im_spade = ax_h.imshow(heatmaps['spade_prob'], cmap=CMAP_VIZ, vmin=0, vmax=1)
ax_h.set_title('(h) Probability Heatmap', fontsize=FONTSIZE)
ax_i.imshow(original_image)
ax_i.imshow(heatmaps['spade_log'], cmap=CMAP_VIZ, vmin=0, vmax=1, alpha=0.5)
ax_i.set_title('(i) Log-density Overlay', fontsize=FONTSIZE)
fig.colorbar(im_spade, cax=cax_spade, label='Normalized Saliency (SPADE)')

for ax in [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g, ax_h, ax_i]:
    ax.axis('off')

fig.tight_layout(rect=[0, 0, 1, 0.96])
output_path_main = PROJECT_ROOT / 'output/model_comparison_main_paper.png'
output_path_main.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path_main, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"✅ Main comparison saved to: {output_path_main}")

-----

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL: Step-by-Step Scanpath Prediction (Corrected Input)
# =================================================================================
import sys
import os
from pathlib import Path
import yaml
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pysaliency
from pysaliency.dataset_config import validation_split

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_scanpath_frozen/dinogaze_spade_v1_sam64_fold0/final_best_val.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1],
    'dino_semantic_feature_layer_idx': -1,
    'num_total_sam_segments': 64,
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'fold': 0,
    'is_scanpath_stage': True,
    'included_fixations': [-1, -2, -3, -4],
}

# --- Prediction Settings ---
MAX_STEPS_TO_SHOW = 9
RANDOM_SEED = 41
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. MANUALLY BUILD THE CONFIG OBJECT ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(
    paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
    stage=SimpleNamespace(
        model_key=MODEL_KEY,
        dataset_key=DATASET_KEY,
        extra={"mit_all_mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}
    )
)
print("✅ Config object created.")

# --- 4. HELPER FUNCTIONS ---
def load_data_for_scanpath_viz(cfg, random_seed):
    print("\n--- Loading data for scanpath visualization ---")
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data")
    ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data

    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0)
    val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)
    
    long_scanpath_found = False
    random.seed(random_seed)
    for _ in range(100):
        image_path = random.choice(val_stim.filenames)
        stimulus_index = stimuli_resized.filenames.index(str(image_path))
        scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
        if len(scanpaths_for_image.xs) > 0 and len(scanpaths_for_image.xs[0]) > MAX_STEPS_TO_SHOW:
            long_scanpath_found = True
            break
    
    if not long_scanpath_found: print("Warning: Could not find a scanpath long enough.")

    print(f"Randomly selected image: {Path(image_path).name}")
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mit_all_mask_dir")
    mask_path = mask_dir / f"{Path(image_path).stem}.png"
    
    gt_scanpath_x = scanpaths_for_image.xs[0]
    gt_scanpath_y = scanpaths_for_image.ys[0]

    return {"image_path": image_path, "mask_path": mask_path, "gt_scanpath_x": gt_scanpath_x, "gt_scanpath_y": gt_scanpath_y}

def load_model_for_visualization(cfg, checkpoint_path, device):
    print(f"\n--- Building model '{cfg.stage.model_key}' from registry ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    print(f"--- Loading checkpoint: {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    missing_keys, unexpected_keys = model.load_state_dict(cleaned_state_dict, strict=False)
    print(f"Loaded weights. Missing: {missing_keys}, Unexpected: {unexpected_keys}")
    model.eval()
    print(f"✅ Model '{cfg.stage.model_key}' built and loaded successfully.")
    return model

# --- 5. EXECUTE LOADING AND PREDICTION ---
viz_data = load_data_for_scanpath_viz(cfg, RANDOM_SEED)
model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)

# Prepare inputs
original_image = Image.open(viz_data['image_path']).convert('RGB')
image_np = np.array(original_image)

# --- FIX: Pass the integer tensor [0, 255] to the model, just like in training ---
image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)

mask_image = Image.open(viz_data['mask_path']).convert('L')
mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)

# Get ground truth scanpath to use for history
gt_x = viz_data['gt_scanpath_x']
gt_y = viz_data['gt_scanpath_y']
history_length = len(cfg.stage.extra['included_fixations'])
saliency_maps_over_time = []

print(f"\n--- Generating {MAX_STEPS_TO_SHOW} one-step-ahead prediction maps ---")
with torch.no_grad():
    for i in range(MAX_STEPS_TO_SHOW):
        # We need history up to fixation `i` to predict fixation `i+1`.
        # The first prediction uses an empty history.
        history_x = gt_x[:i]
        history_y = gt_y[:i]

        current_history_x = history_x[-history_length:]
        current_history_y = history_y[-history_length:]
        padding_needed = history_length - len(current_history_x)
        padded_x = [np.nan] * padding_needed + list(current_history_x)
        padded_y = [np.nan] * padding_needed + list(current_history_y)
        x_hist_tensor = torch.tensor(padded_x, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        y_hist_tensor = torch.tensor(padded_y, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
        log_density_map = model(image_tensor, centerbias, x_hist=x_hist_tensor, y_hist=y_hist_tensor, segmentation_mask=mask_tensor)
        
        prob_map = torch.exp(log_density_map.squeeze()).cpu().numpy()
        saliency_maps_over_time.append({
            'prob_map': prob_map,
            'history_x': history_x,
            'history_y': history_y,
            'next_fix_x': gt_x[i], # The actual fixation the model is trying to predict
            'next_fix_y': gt_y[i]
        })

# --- 6. VISUALIZE THE PREDICTION GRID ---
print("\n--- Creating step-by-step prediction visualization ---")
num_plots = MAX_STEPS_TO_SHOW + 1
grid_cols = 5
grid_rows = math.ceil(num_plots / grid_cols)
fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 5, grid_rows * 5), dpi=100)
axes = axes.flatten()

# Plot 1: Original Image with Full Ground Truth Scanpath
axes[0].imshow(original_image)
axes[0].plot(gt_x, gt_y, 'r-', marker='o', markersize=5, lw=1.5, alpha=0.7)
axes[0].set_title('Original Image & Full GT Scanpath')
axes[0].axis('off')

# Plot subsequent prediction maps
for i in range(MAX_STEPS_TO_SHOW):
    ax = axes[i + 1]
    data = saliency_maps_over_time[i]
    
    ax.imshow(data['prob_map'], cmap='viridis')
    ax.contour(data['prob_map'], levels=np.linspace(data['prob_map'].min(), data['prob_map'].max(), 7), colors='black', linewidths=0.5)

    history_x = data['history_x']
    history_y = data['history_y']
    if len(history_x) > 0:
        ax.plot(history_x, history_y, 'r-', marker='>', markersize=6, lw=1.5)
    
    ax.plot(data['next_fix_x'], data['next_fix_y'], '*', color='white', markersize=12, markeredgecolor='black')
    
    ax.set_title(f'p(f$_{i+1}$ | f$_{{0..{i-1}}}$)')
    ax.axis('off')
    ax.set_aspect('equal')

for i in range(num_plots, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
output_path = PROJECT_ROOT / f'output/{cfg.stage.model_key}_scanpath_steps.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show()
plt.close(fig)
print(f"✅ Step-by-step prediction visualization saved to: {output_path}")

In [ ]:
# =================================================================================
#  FINAL VISUALIZATION CELL: Step-by-Step Scanpath Prediction (Corrected and Simplified)
# =================================================================================
import sys
import os
from pathlib import Path
import yaml
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY, DATA_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_scanpath_frozen/dinogaze_spade_v1_sam64_fold0/final_best_val.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1],
    'dino_semantic_feature_layer_idx': -1,
    'num_total_sam_segments': 64,
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'fold': 0,
    'is_scanpath_stage': True,
    'included_fixations': [-1, -2, -3, -4],
}

# --- Prediction Settings ---
MAX_STEPS_TO_SHOW = 9 # You can keep this high
RANDOM_SEED = 494
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. MANUALLY BUILD THE CONFIG OBJECT ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(
    paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
    stage=SimpleNamespace(
        model_key=MODEL_KEY,
        dataset_key=DATASET_KEY,
        extra={"mit_all_mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}
    )
)
print("✅ Config object created.")

# --- 4. HELPER FUNCTIONS ---
def load_data_for_scanpath_viz(cfg, random_seed):
    print("\n--- Loading data for scanpath visualization ---")
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    logger = logging.getLogger("visualize_data")
    ddp_ctx = MockDDPCtx()
    from src.datasets.mit1003 import _get_mit_data

    stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
    fold = cfg.stage.extra.get("fold", 0)
    val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)

    long_scanpath_found = False
    random.seed(random_seed)
    for _ in range(100):
        image_path = random.choice(val_stim.filenames)
        stimulus_index = stimuli_resized.filenames.index(str(image_path))
        scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
        if len(scanpaths_for_image.xs) > 0 and len(scanpaths_for_image.xs[0]) > MAX_STEPS_TO_SHOW:
            long_scanpath_found = True
            break

    if not long_scanpath_found: print(f"⚠️ Warning: Could not find a scanpath with >{MAX_STEPS_TO_SHOW} fixations. Using one with {len(scanpaths_for_image.xs[0]) if len(scanpaths_for_image.xs) > 0 else 0}.")

    print(f"Randomly selected image: {Path(image_path).name}")
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mit_all_mask_dir")
    mask_path = mask_dir / f"{Path(image_path).stem}.png"

    gt_scanpath_x = scanpaths_for_image.xs[0]
    gt_scanpath_y = scanpaths_for_image.ys[0]

    return {"image_path": image_path, "mask_path": mask_path, "gt_scanpath_x": gt_scanpath_x, "gt_scanpath_y": gt_scanpath_y}

def load_model_for_visualization(cfg, checkpoint_path, device):
    print(f"\n--- Building model '{cfg.stage.model_key}' from registry ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    print(f"--- Loading checkpoint: {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    missing_keys, unexpected_keys = model.load_state_dict(cleaned_state_dict, strict=False)
    print(f"Loaded weights. Missing: {len(missing_keys)} keys, Unexpected: {len(unexpected_keys)} keys")
    model.eval()
    print(f"✅ Model '{cfg.stage.model_key}' built and loaded successfully.")
    return model

# --- 5. EXECUTE LOADING AND PREDICTION ---
viz_data = load_data_for_scanpath_viz(cfg, RANDOM_SEED)
model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)

# Prepare inputs
original_image = Image.open(viz_data['image_path']).convert('RGB')

preprocess = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
image_tensor = preprocess(original_image).unsqueeze(0).to(DEVICE)

mask_image = Image.open(viz_data['mask_path']).convert('L')
mask_np = np.array(mask_image)
mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)

gt_x, gt_y = viz_data['gt_scanpath_x'], viz_data['gt_scanpath_y']
history_length = len(cfg.stage.extra['included_fixations'])
saliency_maps_over_time = []

# ===========================================================================
# --- FIX: Loop only for the number of available fixations ---
# ===========================================================================
with torch.no_grad():
    num_steps_to_run = min(MAX_STEPS_TO_SHOW, len(gt_x))
    print(f"\n--- Ground truth scanpath has {len(gt_x)} fixations. Generating {num_steps_to_run} prediction maps ---")

    for i in range(num_steps_to_run):
        # ===========================================================================
        history_x, history_y = gt_x[:i], gt_y[:i]

        current_history_x = list(history_x[-history_length:])
        current_history_y = list(history_y[-history_length:])
        current_history_x.reverse()
        current_history_y.reverse()
        padding_needed = history_length - len(current_history_x)
        padded_x = current_history_x + [np.nan] * padding_needed
        padded_y = current_history_y + [np.nan] * padding_needed

        x_hist_tensor = torch.tensor(padded_x, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        y_hist_tensor = torch.tensor(padded_y, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=DEVICE)
        log_density_map = model(image_tensor, centerbias, x_hist=x_hist_tensor, y_hist=y_hist_tensor, segmentation_mask=mask_tensor)

        prob_map = torch.exp(log_density_map.squeeze()).cpu().numpy()
        saliency_maps_over_time.append({
            'prob_map': prob_map,
            'history_x': history_x, 'history_y': history_y,
            'next_fix_x': gt_x[i], 'next_fix_y': gt_y[i]
        })

# --- 6. VISUALIZE THE PREDICTION GRID ---
print("\n--- Creating step-by-step prediction visualization ---")
num_plots_to_show = len(saliency_maps_over_time)
num_plots = num_plots_to_show + 1
grid_cols, grid_rows = 5, math.ceil(num_plots / 5)
fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 4, grid_rows * 4), dpi=120)
axes = axes.flatten()

axes[0].imshow(original_image); axes[0].plot(gt_x, gt_y, 'r-', marker='o', markersize=5, lw=1.5, alpha=0.7); axes[0].set_title('Original Image & Full GT Scanpath'); axes[0].axis('off')

for i in range(num_plots_to_show):
    ax, data = axes[i + 1], saliency_maps_over_time[i]
    ax.imshow(data['prob_map'], cmap='viridis'); ax.contour(data['prob_map'], levels=np.linspace(data['prob_map'].min(), data['prob_map'].max(), 7), colors='black', linewidths=0.5, alpha=0.5)
    if len(data['history_x']) > 0: ax.plot(data['history_x'], data['history_y'], 'r-', marker='>', markersize=6, lw=1.5)
    ax.plot(data['next_fix_x'], data['next_fix_y'], '*', color='white', markersize=12, markeredgecolor='black')
    ax.set_title(f'p(f$_{{{i+1}}}$ | f$_{{...{i}}}$)'); ax.axis('off'); ax.set_aspect('equal')

for i in range(num_plots, len(axes)): axes[i].axis('off')
plt.tight_layout(pad=0.5)
output_path = PROJECT_ROOT / f'output/{cfg.stage.model_key}_scanpath_steps_FIXED.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path)
plt.show(); plt.close(fig)
print(f"✅ Step-by-step prediction visualization saved to: {output_path}")

In [ ]:
# =================================================================================
#  BATCH VISUALIZATION CELL: Generate 10 Plots with Different Images (Final Version)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn as nn
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pysaliency
from pysaliency.dataset_config import validation_split
import torchvision.transforms as T
from tqdm import tqdm

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_scanpath_frozen/dinogaze_spade_v1_sam64_fold0/final_best_val.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1],
    'dino_semantic_feature_layer_idx': -1,
    'num_total_sam_segments': 64,
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'fold': 0,
    'is_scanpath_stage': True,
    'included_fixations': [-1, -2, -3, -4],
}

# --- Prediction Settings ---
MAX_STEPS_TO_SHOW = 9  # A good balance for finding enough images and showing the sliding window
NUM_PLOTS_TO_GENERATE = 10
RANDOM_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. MANUALLY BUILD THE CONFIG OBJECT ---
print("\n--- Manually building config object ---")
cfg = SimpleNamespace(
    paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
    stage=SimpleNamespace(
        model_key=MODEL_KEY,
        dataset_key=DATASET_KEY,
        extra={"mit_all_mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}
    )
)
print("✅ Config object created.")

# --- 4. HELPER FUNCTIONS ---
def load_model_for_visualization(cfg, checkpoint_path, device):
    print(f"\n--- Building model '{cfg.stage.model_key}' from registry ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    print(f"--- Loading checkpoint: {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    missing_keys, unexpected_keys = model.load_state_dict(cleaned_state_dict, strict=False)
    print(f"Loaded weights. Missing: {len(missing_keys)} keys, Unexpected: {len(unexpected_keys)} keys")
    model.eval()
    print(f"✅ Model '{cfg.stage.model_key}' built and loaded successfully.")
    return model

def generate_and_save_visualization(model, viz_data, cfg, device, output_path):
    original_image = Image.open(viz_data['image_path']).convert('RGB')
    preprocess = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image_tensor = preprocess(original_image).unsqueeze(0).to(device)
    mask_image = Image.open(viz_data['mask_path']).convert('L')
    mask_np = np.array(mask_image)
    mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(device)
    gt_x, gt_y = viz_data['gt_scanpath_x'], viz_data['gt_scanpath_y']
    history_length = len(cfg.stage.extra['included_fixations'])
    saliency_maps_over_time = []
    with torch.no_grad():
        num_steps_to_run = min(MAX_STEPS_TO_SHOW, len(gt_x))
        for i in range(num_steps_to_run):
            history_x, history_y = gt_x[:i], gt_y[:i]
            current_history_x, current_history_y = list(history_x[-history_length:]), list(history_y[-history_length:])
            current_history_x.reverse(); current_history_y.reverse()
            padding_needed = history_length - len(current_history_x)
            padded_x, padded_y = current_history_x + [np.nan] * padding_needed, current_history_y + [np.nan] * padding_needed
            x_hist_tensor = torch.tensor(padded_x, dtype=torch.float32).unsqueeze(0).to(device)
            y_hist_tensor = torch.tensor(padded_y, dtype=torch.float32).unsqueeze(0).to(device)
            centerbias = torch.zeros((1, image_tensor.shape[2], image_tensor.shape[3]), device=device)
            log_density_map = model(image_tensor, centerbias, x_hist=x_hist_tensor, y_hist=y_hist_tensor, segmentation_mask=mask_tensor)
            prob_map = torch.exp(log_density_map.squeeze()).cpu().numpy()
            saliency_maps_over_time.append({
                'prob_map': prob_map, 'history_x': history_x, 'history_y': history_y,
                'next_fix_x': gt_x[i], 'next_fix_y': gt_y[i]
            })
    num_plots_to_show = len(saliency_maps_over_time)
    num_plots = num_plots_to_show + 1
    grid_cols, grid_rows = 5, math.ceil(num_plots / 5)
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 4, grid_rows * 4), dpi=120)
    axes = axes.flatten()
    axes[0].imshow(original_image)
    axes[0].plot(gt_x, gt_y, 'r-', marker='o', markersize=5, lw=1.5, alpha=0.7)
    axes[0].set_title(f'GT Scanpath ({Path(viz_data["image_path"]).name})')
    axes[0].axis('off')
    for i in range(num_plots_to_show):
        ax, data = axes[i + 1], saliency_maps_over_time[i]
        ax.imshow(data['prob_map'], cmap='viridis')
        ax.contour(data['prob_map'], levels=np.linspace(data['prob_map'].min(), data['prob_map'].max(), 7), colors='black', linewidths=0.5, alpha=0.5)
        if len(data['history_x']) > 0: ax.plot(data['history_x'], data['history_y'], 'r-', marker='>', markersize=6, lw=1.5)
        
        if i > 0:
            ax.plot(data['next_fix_x'], data['next_fix_y'], '*', color='white', markersize=12, markeredgecolor='black')
        
        # --- FINAL: Title logic that reflects the model's limited memory ---
        if i == 0:
            title = f'p(f$_{{1}}$) | Initial Saliency'
        else:
            history_start_index = max(1, i - history_length + 1)
            if history_start_index == i:
                history_str = f'f$_{{{i}}}$'
            else:
                history_str = f'f$_{{{history_start_index}..{i}}}$'
            title = f'p(f$_{{{i+1}}}$ | {history_str})'
        ax.set_title(title)
        
        ax.axis('off'); ax.set_aspect('equal')
    for i in range(num_plots, len(axes)): axes[i].axis('off')
    plt.tight_layout(pad=0.5)
    plt.savefig(output_path)
    plt.close(fig)
    print(f"✅ Plot saved to: {output_path}")

# ================================================================================
# --- 5. SETUP: LOAD DATA AND MODEL ONCE ---
# ================================================================================

model = load_model_for_visualization(cfg, CHECKPOINT_PATH, DEVICE)

print("\n--- Pre-loading dataset to find valid images ---")
class MockDDPCtx:
    is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
    def barrier(self): pass
logger = logging.getLogger("visualize_data")
ddp_ctx = MockDDPCtx()
from src.datasets.mit1003 import _get_mit_data

stimuli_resized, scanpaths_resized = _get_mit_data(cfg, ddp_ctx, logger)
fold = cfg.stage.extra.get("fold", 0)
val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold)

valid_images_for_viz = []
print(f"--- Searching {len(val_stim.filenames)} validation images for long scanpaths... ---")
for image_path in tqdm(val_stim.filenames):
    stimulus_index = stimuli_resized.filenames.index(str(image_path))
    scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    if len(scanpaths_for_image.xs) > 0 and len(scanpaths_for_image.xs[0]) > MAX_STEPS_TO_SHOW:
        valid_images_for_viz.append(image_path)
        
print(f"Found {len(valid_images_for_viz)} valid images with >{MAX_STEPS_TO_SHOW} fixations.")

if len(valid_images_for_viz) < NUM_PLOTS_TO_GENERATE:
    print(f"⚠️ WARNING: Found only {len(valid_images_for_viz)} valid images, but requested {NUM_PLOTS_TO_GENERATE}. Will only generate {len(valid_images_for_viz)} plots.")
    NUM_PLOTS_TO_GENERATE = len(valid_images_for_viz)

random.seed(RANDOM_SEED)
random.shuffle(valid_images_for_viz)

# ================================================================================
# --- 6. BATCH GENERATION SCRIPT ---
# ================================================================================

output_dir = PROJECT_ROOT / f'output/{cfg.stage.model_key}_batch_viz'
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n--- Generating {NUM_PLOTS_TO_GENERATE} unique plots ---")

for i in range(NUM_PLOTS_TO_GENERATE):
    print(f"\n{'='*20} PLOT {i+1}/{NUM_PLOTS_TO_GENERATE} {'='*20}")
    
    image_path = valid_images_for_viz[i]
    print(f"Processing image: {Path(image_path).name}")
    
    stimulus_index = stimuli_resized.filenames.index(str(image_path))
    scanpaths_for_image = scanpaths_resized.scanpaths[scanpaths_resized.scanpaths.n == stimulus_index]
    mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mit_all_mask_dir")
    mask_path = mask_dir / f"{Path(image_path).stem}.png"
    
    viz_data = {
        "image_path": image_path,
        "mask_path": mask_path,
        "gt_scanpath_x": scanpaths_for_image.xs[0],
        "gt_scanpath_y": scanpaths_for_image.ys[0],
    }
            
    output_path = output_dir / f'scanpath_plot_{i+1:02d}_{Path(viz_data["image_path"]).stem}.png'
    generate_and_save_visualization(model, viz_data, cfg, DEVICE, output_path)

print(f"\n\n✅✅✅ All {NUM_PLOTS_TO_GENERATE} plots have been generated in the directory: {output_dir} ✅✅✅")

In [ ]:
# = =================================================================================
#  CONTROL EXPERIMENTS SCRIPT for DinoGaze-SPADE (Corrected v1.3)
# =================================================================================
import sys
import os
from pathlib import Path
import random
import logging
from types import SimpleNamespace

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# --- FIX: Import the class directly from the top-level pysaliency module ---
import pysaliency
from pysaliency import CenterBias 
from pysaliency.dataset_config import validation_split

import torchvision.transforms as T
from tqdm import tqdm

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

from src.registry import MODEL_REGISTRY
from src.train import _auto_import_modules
from src.metrics import log_likelihood

_auto_import_modules()
print("✅ Registries populated.")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/dinogaze_spade_v1_sam64_fold0/step-0005.pth'
MIT_MASK_DIR = 'masks/mit1003/sam_vitl_k64_mit'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'
DATASET_KEY = 'MIT1003'

# --- Model & Data Parameters (Must match the trained model) ---
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1],
    'dino_semantic_feature_layer_idx': -1,
    'num_total_sam_segments': 64,
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'fold': 0,
}

# --- General Settings ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================

# --- 3. HELPER FUNCTIONS ---

def build_config():
    """Builds the configuration object."""
    return SimpleNamespace(
        paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
        stage=SimpleNamespace(
            model_key=MODEL_KEY,
            dataset_key=DATASET_KEY,
            extra={"mit_all_mask_dir": MIT_MASK_DIR, **MODEL_PARAMS}
        )
    )

def load_model(cfg, checkpoint_path, device):
    """Loads the specified model from a checkpoint."""
    print(f"\n--- Loading Model: {cfg.stage.model_key} ---")
    model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_state_dict = checkpoint.get('model', checkpoint)
    cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
    model.load_state_dict(cleaned_state_dict, strict=False)
    model.eval()
    print("✅ Model loaded successfully.")
    return model

def get_dataset_info(cfg):
    """Loads dataset file lists and calculates the baseline performance."""
    print("\n--- Loading Dataset Information ---")
    class MockDDPCtx:
        is_master = True; enabled = False; rank = 0; world = 1; device = 'cpu'
        def barrier(self): pass
    from src.datasets.mit1003 import _get_mit_data
    
    stimuli, scanpaths = _get_mit_data(cfg, MockDDPCtx(), logging.getLogger())
    fold = cfg.stage.extra.get("fold", 0)
    val_stimuli, val_scanpaths = validation_split(stimuli, scanpaths, crossval_folds=10, fold_no=fold)

    # Calculate baseline Log-Likelihood for IG
    # --- FIX: Use the top-level CenterBias class ---
    center_bias_model = CenterBias(val_stimuli, val_scanpaths)
    baseline_ll = center_bias_model.log_likelihood(val_stimuli, val_scanpaths)
    
    print(f"Found {len(val_stimuli.filenames)} validation images.")
    print(f"Baseline Log-Likelihood for this fold: {baseline_ll:.4f}")
    
    return val_stimuli, val_scanpaths, baseline_ll

def fixations_to_tensor(xs, ys, shape):
    """Converts fixation coordinates to a dense tensor mask."""
    if not len(xs):
        return torch.zeros(shape, dtype=torch.float32)
    
    xs = np.clip(np.round(xs).astype(int), 0, shape[1] - 1)
    ys = np.clip(np.round(ys).astype(int), 0, shape[0] - 1)
    
    coords = np.stack([ys, xs], axis=0)
    indices = torch.from_numpy(coords)
    values = torch.ones(len(xs), dtype=torch.float32)
    sparse_map = torch.sparse_coo_tensor(indices, values, shape, dtype=torch.float32)
    return sparse_map.to_dense()

# ================================================================================
# --- 4. MAIN EXECUTION ---
# ================================================================================

cfg = build_config()
model = load_model(cfg, CHECKPOINT_PATH, DEVICE)
val_stimuli, val_scanpaths, baseline_ll = get_dataset_info(cfg)

preprocess = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

mask_dir = PROJECT_ROOT / cfg.stage.extra.get("mit_all_mask_dir")
all_mask_paths = [mask_dir / f"{Path(f).stem}.png" for f in val_stimuli.filenames]

# --- CONTROL 1: ZERO-INFORMATION ---
print("\n" + "="*20 + " Running Zero-Information Control " + "="*20)
zero_info_lls = []
with torch.no_grad():
    for i in tqdm(range(len(val_stimuli.filenames)), desc="Zero-Info Control"):
        image_path = val_stimuli.filenames[i]
        original_image = Image.open(image_path).convert('RGB')
        image_tensor = preprocess(original_image).unsqueeze(0).to(DEVICE)
        
        zero_mask = torch.zeros(1, original_image.height, original_image.width, dtype=torch.long, device=DEVICE)
        
        fix_indices = np.where(val_scanpaths.n == i)[0]
        fix_xs, fix_ys = val_scanpaths.x[fix_indices], val_scanpaths.y[fix_indices]
        fixation_map = fixations_to_tensor(fix_xs, fix_ys, (original_image.height, original_image.width)).to(DEVICE)

        centerbias = torch.zeros_like(zero_mask, dtype=torch.float32)
        log_density = model(image_tensor, centerbias, segmentation_mask=zero_mask)
        
        ll = log_likelihood(log_density, fixation_map).item()
        zero_info_lls.append(ll)

avg_ll_zero = np.mean(zero_info_lls)
ig_zero = avg_ll_zero - baseline_ll
print("\n--- Zero-Information Control Results ---")
print(f"Average Log-Likelihood: {avg_ll_zero:.4f}")
print(f"Information Gain (IG): {ig_zero:.4f}")
print(f"Formatted for paper: '...performance dropped to {ig_zero:.2f} IG...'")

# --- CONTROL 2: MISMATCHED-INFORMATION ---
print("\n" + "="*20 + " Running Mismatched-Information Control " + "="*20)
mismatched_lls = []
shuffled_mask_paths = random.sample(all_mask_paths, len(all_mask_paths))
with torch.no_grad():
    for i in tqdm(range(len(val_stimuli.filenames)), desc="Mismatched-Info Control"):
        if all_mask_paths[i] == shuffled_mask_paths[i]:
            j = (i + 1) % len(all_mask_paths)
            shuffled_mask_paths[i], shuffled_mask_paths[j] = shuffled_mask_paths[j], shuffled_mask_paths[i]
        
        image_path = val_stimuli.filenames[i]
        original_image = Image.open(image_path).convert('RGB')
        image_tensor = preprocess(original_image).unsqueeze(0).to(DEVICE)
        
        mismatched_mask_path = shuffled_mask_paths[i]
        mask_image = Image.open(mismatched_mask_path).convert('L')
        mask_tensor = torch.from_numpy(np.array(mask_image)).long().unsqueeze(0).to(DEVICE)

        fix_indices = np.where(val_scanpaths.n == i)[0]
        fix_xs, fix_ys = val_scanpaths.x[fix_indices], val_scanpaths.y[fix_indices]
        fixation_map = fixations_to_tensor(fix_xs, fix_ys, (original_image.height, original_image.width)).to(DEVICE)
        
        centerbias = torch.zeros_like(mask_tensor, dtype=torch.float32)
        log_density = model(image_tensor, centerbias, segmentation_mask=mask_tensor)
        
        ll = log_likelihood(log_density, fixation_map).item()
        mismatched_lls.append(ll)

avg_ll_mismatched = np.mean(mismatched_lls)
ig_mismatched = avg_ll_mismatched - baseline_ll
print("\n--- Mismatched-Information Control Results ---")
print(f"Average Log-Likelihood: {avg_ll_mismatched:.4f}")
print(f"Information Gain (IG): {ig_mismatched:.4f}")
print(f"Formatted for paper: '...performance plummeted to {ig_mismatched:.2f} IG...'")

In [1]:
# =================================================================================
# --- INTERACTIVE EMBEDDING EXPLORATION (YAML-Free & Self-Contained) ---
# This version uses a manual config to load the model (no YAML file needed)
# and provides an interactive UMAP plot to explore segment embeddings.
# =================================================================================

# --- 0. IMPORTS & SETUP ---
import numpy as np
import torch
import random
import os
import sys
import logging
from pathlib import Path
from types import SimpleNamespace
from tqdm.notebook import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, clear_output
import umap
import cloudpickle as cpickle
import torch.nn.functional as F
import pysaliency
from pysaliency.dataset_config import validation_split

# This magic command is essential for interactive plots. Place it at the top.
%matplotlib ipympl

# --- 1. SET UP PYTHON PATH AND REGISTRIES ---
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd() # Fallback if notebook is not in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added '{PROJECT_ROOT}' to Python's path.")

# Dynamically import models to populate the registry
from src.registry import MODEL_REGISTRY
from src.train import _auto_import_modules
_auto_import_modules()
print("✅ Registries populated.")

# Setup basic logging and torch_scatter check
logging.basicConfig(level=logging.INFO, format="[%(asctime)s][%(name)s][%(levelname)s] - %(message)s")
logger = logging.getLogger("EmbeddingExplorer")
try:
    from torch_scatter import scatter_mean
except ImportError:
    raise ImportError("torch_scatter is required for this visualization. Please install it (`pip install torch_scatter`).")

# ================================================================================
# --- 2. USER CONFIGURATION (EDIT THIS SECTION) ---
# ================================================================================
# --- File Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / 'experiments/mit_spatial_finetune/dinogaze_spade_v1_sam64_fold0/final_best_val.pth'

# --- Registry Keys ---
MODEL_KEY = 'dinogaze_spade_v1'

# --- Model & Data Parameters ---
# These parameters must match the configuration used for training the checkpoint.
MODEL_PARAMS = {
    'dino_model_name': 'dinov2_vitl14',
    'dino_patch_size': 14,
    'dino_layers_for_main_path': [-3, -2, -1],
    'dino_semantic_feature_layer_idx': -1,
    'num_total_segments': 64,  # Matches `sam64` in your other example
    'finalizer_initial_sigma': 8.0,
    'finalizer_learn_sigma': True,
    'mask_dir': 'masks/mit1003/sam_vitl_k64_mit', # Relative to PROJECT_ROOT
    'fold': 0,
    'is_scanpath_stage': False, # Important for spatial-only models
    'requires_segmentation': True
}

# --- Visualization Settings ---
NUM_IMAGES_TO_PROCESS = 100
N_NEIGHBORS_TO_SHOW = 9
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================================================================


# --- 3. BUILD CONFIG, MODEL, AND LOAD DATA ---
logger.info("--- Building config, model, and loading data ---")

# 3a. Manually build the config object to pass to the model builder
cfg = SimpleNamespace(
    paths={"dataset_dir": PROJECT_ROOT / "data/pysaliency_datasets"},
    stage=SimpleNamespace(
        model_key=MODEL_KEY,
        extra=MODEL_PARAMS
    )
)

# 3b. Build and load the model
model = MODEL_REGISTRY[cfg.stage.model_key](cfg).to(DEVICE)
logger.info(f"Loading weights from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model_state_dict = checkpoint.get('model', checkpoint)
# Clean 'module.' prefix if model was saved from DDP
cleaned_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
model.load_state_dict(cleaned_state_dict, strict=False)
model.eval()
logger.info("✅ Model loaded and in evaluation mode.")

# 3c. Load the dataset stimuli and scanpaths
data_cache_dir = cfg.paths["dataset_dir"] / "MIT1003_converted_cache"
stimuli_cache_path = data_cache_dir / "stimuli.pkl"
scanpaths_cache_path = data_cache_dir / "scanpaths.pkl"
if not stimuli_cache_path.exists() or not scanpaths_cache_path.exists():
    raise FileNotFoundError(f"Could not find data caches in {data_cache_dir}. Please run a training first.")

with open(stimuli_cache_path, "rb") as f: stimuli_resized = cpickle.load(f)
with open(scanpaths_cache_path, "rb") as f: scanpaths_resized = cpickle.load(f)

fold_no = cfg.stage.extra.get('fold', 0)
val_stim, _ = validation_split(stimuli_resized, scanpaths_resized, crossval_folds=10, fold_no=fold_no)
image_files = val_stim.filenames
random.shuffle(image_files)
logger.info(f"✅ Data loaded for fold {fold_no}. Found {len(image_files)} validation images.")


# --- 4. EMBEDDING EXTRACTION & DATA COLLECTION ---
def extract_segment_embeddings(model, image_tensor, mask_tensor):
    """Helper function to extract mean DINO embeddings for each segment."""
    with torch.no_grad():
        extracted_feature_maps = model.features(image_tensor)
        F_semantic_patches = extracted_feature_maps[model.semantic_feature_layer_idx]
        B, C_dino, H_p, W_p = F_semantic_patches.shape
        segmap_at_feat_res = F.interpolate(mask_tensor.unsqueeze(1).float(), size=(H_p, W_p), mode='nearest').long()
        flat_features = F_semantic_patches.permute(0, 2, 3, 1).reshape(-1, C_dino)
        flat_segmap_at_feat_res = segmap_at_feat_res.view(-1)
        batch_idx_tensor = torch.arange(B, device=DEVICE).view(B, 1).expand(-1, H_p * W_p).reshape(-1)
        global_segment_ids = batch_idx_tensor * model.num_total_segments + torch.clamp(flat_segmap_at_feat_res, 0, model.num_total_segments - 1)
        segment_avg_features = scatter_mean(src=flat_features, index=global_segment_ids, dim=0, dim_size=B * model.num_total_segments)
        segment_avg_features = torch.nan_to_num(segment_avg_features, nan=0.0)
        return segment_avg_features.view(B, model.num_total_segments, C_dino)

logger.info(f"Starting embedding extraction from {NUM_IMAGES_TO_PROCESS} random images...")
all_embeddings, metadata = [], []
mask_dir = PROJECT_ROOT / cfg.stage.extra['mask_dir']

for image_path in tqdm(image_files[:NUM_IMAGES_TO_PROCESS]):
    try:
        img_pil = Image.open(image_path).convert("RGB")
        img_np = np.array(img_pil)
        # Simple ToTensor conversion and add batch dimension
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE).float() / 255.0

        mask_stem = Path(image_path).stem
        mask_path = mask_dir / f"{mask_stem}.png"
        mask_pil = Image.open(mask_path).convert("L")
        if mask_pil.size != img_pil.size:
            mask_pil = mask_pil.resize(img_pil.size, Image.Resampling.NEAREST)
        mask_np = np.array(mask_pil)
        mask_tensor = torch.from_numpy(mask_np).long().unsqueeze(0).to(DEVICE)

        embeddings_for_image = extract_segment_embeddings(model, img_tensor, mask_tensor).squeeze(0)
        present_segments = torch.unique(mask_tensor).cpu().numpy()

        for seg_id in present_segments:
            if seg_id >= model.num_total_segments: continue
            embedding = embeddings_for_image[seg_id].cpu().numpy()
            if np.all(embedding == 0): continue

            all_embeddings.append(embedding)

            # Create a cropped RGBA image of the segment for visualization
            segment_pixels = np.where(mask_np == seg_id)
            if segment_pixels[0].size > 0:
                ymin, ymax = segment_pixels[0].min(), segment_pixels[0].max()
                xmin, xmax = segment_pixels[1].min(), segment_pixels[1].max()
                segment_img_rgba = Image.fromarray(img_np).convert("RGBA")
                mask_for_alpha = np.zeros_like(mask_np, dtype=np.uint8)
                mask_for_alpha[segment_pixels] = 255
                segment_img_rgba.putalpha(Image.fromarray(mask_for_alpha))
                cropped_segment = segment_img_rgba.crop((xmin, ymin, xmax, ymax))
                metadata.append({'image_path': image_path, 'segment_id': seg_id, 'segment_image': cropped_segment})
    except Exception as e:
        logger.warning(f"Skipping {Path(image_path).name} due to error: {e}")


# --- 5. DIMENSIONALITY REDUCTION & INTERACTIVE VISUALIZATION ---
MIN_SAMPLES_FOR_UMAP = 20
if len(all_embeddings) < MIN_SAMPLES_FOR_UMAP:
    logger.error(f"Not enough valid embeddings ({len(all_embeddings)}). Need at least {MIN_SAMPLES_FOR_UMAP}.")
else:
    all_embeddings = np.array(all_embeddings)
    logger.info(f"Extracted {len(all_embeddings)} segment embeddings. Performing UMAP reduction...")
    n_neighbors_for_umap = min(15, len(all_embeddings) - 1)
    reducer = umap.UMAP(n_neighbors=n_neighbors_for_umap, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(all_embeddings)
    logger.info("✅ UMAP complete. Creating interactive plot...")

    nn_model = NearestNeighbors(n_neighbors=min(N_NEIGHBORS_TO_SHOW, len(embeddings_2d)), metric='euclidean')
    nn_model.fit(embeddings_2d)

    fig_main, ax_main = plt.subplots(figsize=(10, 8))
    colors = embeddings_2d[:, 1] # Color by y-axis value for visual structure
    scatter = ax_main.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=5, alpha=0.6, c=colors, cmap='viridis')
    ax_main.set_title("UMAP Projection of Segment Embeddings")
    fig_main.canvas.header_visible = False # Cleaner plot in VSCode/JupyterLab

    out_neighbors = widgets.Output()

    def on_click(event):
        if event.inaxes != ax_main: return
        click_coords = np.array([[event.xdata, event.ydata]])
        distances, indices = nn_model.kneighbors(click_coords)
        with out_neighbors:
            clear_output(wait=True)
            grid_cols = 3
            grid_rows = int(np.ceil(N_NEIGHBORS_TO_SHOW / grid_cols))
            fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(12, 4 * grid_rows))
            axes = axes.flatten()
            fig.suptitle("Nearest Neighbors in Embedding Space", fontsize=16)

            for i, idx in enumerate(indices[0]):
                meta = metadata[idx]
                ax = axes[i]
                ax.imshow(meta['segment_image'])
                ax.set_title(f"{Path(meta['image_path']).name}\nSeg ID: {meta['segment_id']}", fontsize=8)
                ax.axis('off')

            for i in range(len(indices[0]), len(axes)):
                axes[i].axis('off') # Hide unused subplots

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

    fig_main.canvas.mpl_connect('button_press_event', on_click)

    # Display the output widget where neighbor images will appear
    display(out_neighbors)

/home/omirako/Documents/Decoding_Neural_Dynamics_of_Visual_Perceptual_Segmentation/.pixi/envs/default/lib/python3.11/site-packages/pysaliency/external_models/matlab_models.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


Added '/home/omirako/Documents/Decoding_Neural_Dynamics_of_Visual_Perceptual_Segmentation' to Python's path.


[2025-10-31 16:09:31,076][EmbeddingExplorer][INFO] - --- Building config, model, and loading data ---
[2025-10-31 16:09:31,077][src.models.dinogaze_spade_v1][INFO] - Building DinoGazeSpade (v1 architecture) with configuration: {'dino_model_name': 'dinov2_vitl14', 'dino_patch_size': 14, 'dino_layers_for_main_path': [-3, -2, -1], 'dino_semantic_feature_layer_idx': -1, 'num_total_segments': 64, 'finalizer_initial_sigma': 8.0, 'finalizer_learn_sigma': True, 'mask_dir': 'masks/mit1003/sam_vitl_k64_mit', 'fold': 0, 'is_scanpath_stage': False, 'requires_segmentation': True}


✅ Registries populated.


Using cache found in /home/omirako/.cache/torch/hub/facebookresearch_dinov2_main
/home/omirako/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/omirako/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/omirako/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
[2025-10-31 16:09:31,483][dinov2][INFO] - using MLP layer as FFN
[2025-10-31 16:09:34,866][src.models.dinogaze_spade_v1][INFO] -   - Building in SPATIAL-ONLY mode.
[2025-10-31 16:09:35,173][EmbeddingExplorer][INFO] - Loading weights from: /home/omirako/Documents/Decoding_Neural_Dynamics_of_Visual_Perceptual_Segmentation/experiments/mit_spatial_finetune/dinogaze

FileNotFoundError: [Errno 2] No such file or directory: '/home/omirako/Documents/Decoding_Neural_Dynamics_of_Visual_Perceptual_Segmentation/experiments/mit_spatial_finetune/dinogaze_spade_v1_sam64_fold0/final_best_val.pth'